# 社交媒体上的“情感极化”与“传播热度”分析

本项目分析Reddit气候与能源议题讨论中的情绪表达、情感极化与传播热度之间的关系。数据集来源为项目文件夹中的`主数据集/comment.parquet`，内容为Reddit气候与能源相关subreddit的帖子、评论和回复数据。主流程从该主数据集开始，依次完成评论展开、文本清洗、GoEmotions情绪识别、TF-IDF+NMF主题建模、帖子层极化指标构造、传播热度建模数据集生成、统计建模、机器学习训练、稳健性检验和报告材料生成。

开源仓库：<https://github.com/yie793379z-prog/reddit-climate-emotion-polarization>。仓库中可以查看本notebook、README、图表结果和结果表；主数据文件和本地情绪模型权重不直接上传，数据来源、模型来源和本地放置路径均在README中说明。

读者拿到主数据集、GoEmotions文件、本地情绪模型和本notebook后，可以在自己的电脑上从上到下完整运行。

研究解释边界需要提前说明：Redditupvotes、评论数和回复数是平台内部的互动指标，不能直接等同于真实曝光量或公共影响力。统计模型用于解释相关关系，机器学习模型用于解释预测模式，二者都不直接支持因果判断。

## 运行说明

运行前请确认项目根目录中至少包含`full_analysis_pipeline.ipynb`、`主数据集/comment.parquet`、`GoEmotions/`和`model_cache/roberta-base-go_emotions/`。这些文件覆盖主数据、情绪标签、情绪映射和本地模型权重，是完整运行的必要输入。

建议使用Python3.10或Python3.11，并在JupyterNotebook或VSCodeNotebook中从上到下运行全部单元。若某个依赖包缺失，根据报错安装对应包即可。Notebook内所有路径都基于项目根目录计算，不需要手动改成绝对路径。

运行逻辑如下：

1.如果某些中间parquet已经存在，Notebook会优先读取正式中间结果，减少重复计算时间。
2.如果关键中间文件不存在，Notebook会从主数据和前序步骤重新生成。
3.情绪识别步骤最耗时，因为需要对评论和回复文本进行本地模型推理。
4.描述统计表先生成，统计建模、机器学习训练和稳健性检验随后执行。
5.所有正式图表放在第12节统一导出，避免还没完成分析就提前画结果图。
6.最后的完整性检查只负责确认输入、处理结果和输出文件是否齐全，不替代前面的分析步骤。


In [ ]:
# 本代码块目标：集中配置项目运行所需的Python库、路径、随机种子和通用保存函数。
# 输入：项目根目录、主数据路径、GoEmotions目录、本地情绪模型目录。
# 输出：后续单元共享的路径常量、随机种子、目录对象和基础工具函数。
# 处理思路：所有路径都从当前工作目录推导，避免写死个人电脑上的绝对路径。

# 先导入路径库。Path比字符串路径更稳定，后续拼接目录时不容易出错。
from pathlib import Path
# os和sys用于读取运行环境信息；这里只打印环境，不把环境写死进分析。
import os
import sys
import json
import ast
import re
import math
import warnings
from itertools import combinations

# numpy负责数值计算，pandas负责表格处理，这两者是后续所有分析的基础。
import numpy as np
import pandas as pd

# BASE_DIR固定为当前项目根目录。
# 后续所有文件都从这里出发，别人移动整个文件夹后也不需要改绝对路径。
BASE_DIR = Path('.')
# 主数据是唯一的帖子级输入，后面评论表、回复表、建模表都由它展开或派生。
MAIN_DATA_PATH = BASE_DIR / '主数据集' / 'comment.parquet'
# GoEmotions目录保存标签名和情绪映射，用来解释模型输出的28个概率列。
GOEMOTIONS_DIR = BASE_DIR / 'GoEmotions'
# 本地模型目录保存tokenizer和权重，运行时不需要从网络下载模型。
MODEL_DIR = BASE_DIR / 'model_cache' / 'roberta-base-go_emotions'
PROCESSED_DIR = BASE_DIR / 'processed_data'
OUTPUT_TABLE_DIR = BASE_DIR / 'outputs' / 'tables'
OUTPUT_FIGURE_DIR = BASE_DIR / 'outputs' / 'figures'

# 三个输出目录分别存放中间parquet、结果表和最终图表。
# exist_ok=True表示目录已存在时不报错，方便多次运行同一notebook。
for d in [PROCESSED_DIR, OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 下面这些路径是各阶段的稳定产物名。
# 稳定命名的好处是每个阶段都能判断“已有结果能否直接读取”。
COMMENTS_FLAT_PATH = PROCESSED_DIR / 'comments_flat_raw.parquet'
REPLIES_FLAT_PATH = PROCESSED_DIR / 'replies_flat_raw.parquet'
COMMENTS_CLEAN_PATH = PROCESSED_DIR / 'comments_clean.parquet'
REPLIES_CLEAN_PATH = PROCESSED_DIR / 'replies_clean.parquet'
COMMENTS_EMOTION_PATH = PROCESSED_DIR / 'comments_with_emotions.parquet'
REPLIES_EMOTION_PATH = PROCESSED_DIR / 'replies_with_emotions.parquet'
COMMENTS_TOPIC_PATH = PROCESSED_DIR / 'comments_with_topics.parquet'
POST_POLARIZATION_PATH = PROCESSED_DIR / 'post_level_polarization.parquet'
COMMENT_MODEL_PATH = PROCESSED_DIR / 'modeling_dataset_comment_level.parquet'
POST_MODEL_PATH = PROCESSED_DIR / 'modeling_dataset_post_level.parquet'

# 情绪识别最耗时。
# 如果情绪结果缺失，RUN_EMOTION_IF_MISSING=True会允许notebook重新推理。
RUN_EMOTION_IF_MISSING = True
# BATCH_SIZE和MAX_LENGTH控制本地情绪模型推理的速度和内存占用。
# 数值保守一些，可以让普通笔记本电脑更稳地跑完。
BATCH_SIZE = 8
MAX_LENGTH = 128
CHUNK_SIZE = 2000
# RANDOM_STATE统一控制抽样、主题模型、机器学习划分等随机过程。
# 这样不同步骤不会各自使用隐含随机状态，结果更容易比较。
RANDOM_STATE = 42

# 打印基础运行信息，便于读者排查内核或工作目录是否选错。
print('Python:', sys.version)
print('Working directory:', Path.cwd())
print('All paths are relative to:', BASE_DIR.resolve())

In [ ]:
# 本代码块目标：定义文件检查、文本预览、编码保存和数值转换等通用工具函数。
# 输入：任意路径、DataFrame对象、Series对象或原始字段值。
# 输出：统一的文件大小、预览表、UTF-8-SIG表格和安全数值字段。
# 处理思路：把重复的检查逻辑集中到函数中，减少后续单元的样板代码。

def file_size_mb(path):
    # 文件大小用于判断文件是否真实存在且内容非空。
    # 返回MB比返回字节数更适合人工快速阅读。
    path = Path(path)
    return round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else np.nan

def require_file(path, label=None):
    # 所有必要输入都先经过require_file。
    # 如果文件缺失，立刻抛错，比后面出现模糊的读取错误更容易定位问题。
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"缺少必要文件: {label or path}")
    return path

def save_csv(df, path):
    # 结果表统一用UTF-8-SIG保存。
    # 这样既能被Python稳定读取，也能在Excel中较少出现中文乱码。
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'Saved: {path}')

def save_parquet(df, path):
    # parquet保存中间数据，优点是保留字段类型、读取快、文件也比CSV更紧凑。
    # 每次保存时打印行数，方便确认阶段产物规模是否异常。
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    print(f'Saved: {path}  rows={len(df):,}')

def safe_to_numeric(s):
    # 社交媒体数据里互动数字可能混有空值或字符串。
    # errors='coerce'把异常值转为NaN，后续再按场景填充或删除。
    return pd.to_numeric(s, errors='coerce')

def safe_to_datetime(s):
    # 时间字段统一转成UTC时间，避免不同本地时区影响时间比较。
    return pd.to_datetime(s, errors='coerce', utc=True)

def clean_text_basic(text):
    # 基础清洗只做空值处理、首尾空白处理和连续空白合并。
    # 这里不做词干化或激进删除，避免过早损失语义信息。
    if pd.isna(text):
        return ''
    text = str(text).strip()
    return re.sub(r'\s+', ' ', text)

def is_deleted_or_removed(text):
    # Reddit常见占位文本不能代表真实表达，应在建模前剔除。
    return clean_text_basic(text).lower() in {'[deleted]', '[removed]', 'deleted', 'removed'}

def contains_us_political_keywords(text):
    # 原始语料中可能混入明显美国党派政治讨论。
    # 这些文本会偏离气候与能源传播主题，因此作为质量控制标记剔除。
    keywords = ['trump','biden','democrat','democrats','republican','republicans','election','elections','maga','white house','gop','liberal','conservative']
    t = clean_text_basic(text).lower()
    return any(k in t for k in keywords)

def text_features(series):
    # 文本长度、词数、URL、问号和感叹号属于表达形式特征。
    # 它们作为表达形式特征，后面可用于控制文本形态对热度的影响。
    text = series.fillna('').astype(str)
    return pd.DataFrame({
        'text_length_chars': text.str.len(),
        'word_count': text.str.split().str.len().fillna(0).astype(int),
        'has_url': text.str.contains(r'https?://|www\.', case=False, regex=True, na=False),
        'has_question': text.str.contains(r'\?', regex=True, na=False),
        'has_exclamation': text.str.contains(r'!', regex=True, na=False),
    })

def normalize_nested_object(x):
    # comments字段可能是list、dict、字符串化JSON或空值。
    # 这个函数把不同形态统一转成列表，降低展开嵌套结构时的分支复杂度。
    if x is None:
        return []
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, dict):
        return [x]
    if isinstance(x, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                y = parser(x)
                return normalize_nested_object(y)
            except Exception:
                continue
        return []
    return []

def get_first_existing(d, candidates, default=None):
    # 不同来源可能把同一字段写成不同大小写或不同命名。
    # 候选字段按优先级查找，保证展开时尽量拿到可用值。
    if not isinstance(d, dict):
        return default
    for c in candidates:
        v = d.get(c, default)
        if v is not None and not (isinstance(v, float) and np.isnan(v)) and v != '':
            return v
    return default

def zscore(s):
    # zscore用于回归模型前的标准化。
    # 如果变量没有方差，直接返回0，避免除以0导致整列NaN。
    s = pd.to_numeric(s, errors='coerce')
    sd = s.std(skipna=True)
    if pd.isna(sd) or sd == 0:
        return pd.Series(0, index=s.index)
    return (s - s.mean(skipna=True)) / sd

## 1.数据读取与主数据结构检查

本节读取`主数据集/comment.parquet`，确认原始帖子数据存在、能够被pandas读取，并查看字段结构。数据集来源说明：该文件是项目提供的Reddit气候与能源相关subreddit帖子数据，每一行代表一个帖子，`comments`字段保存该帖子下的嵌套评论和回复。后续所有评论级、回复级和帖子级数据都从这里展开。

In [ ]:
# 本代码块目标：读取主数据并检查帖子级字段结构。
# 输入：主数据文件`主数据集/comment.parquet`。
# 输出：原始帖子DataFrame、字段预览、数据规模和comments字段样例。
# 处理思路：先确认文件可读，再观察嵌套评论字段的真实形态。

# 先确认主数据存在。
# 如果这里失败，后面所有步骤都没有可靠输入，因此应立即停止。
require_file(MAIN_DATA_PATH, '主数据集/comment.parquet')
# 读取帖子级parquet。
# 一行代表一个帖子，comments字段仍然是嵌套结构，下一节才展开。
main_df = pd.read_parquet(MAIN_DATA_PATH)
print('main_df shape:', main_df.shape)
# 预览前几行用于确认字段是否和预期一致。
display(main_df.head())

# 同时检查GoEmotions标签文件。
# 后续模型输出概率列必须能和这些标签一一对应。
go_labels_path = require_file(GOEMOTIONS_DIR / 'emotions.txt', 'GoEmotions/emotions.txt')
go_labels = [line.strip() for line in go_labels_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print('GoEmotions labels:', len(go_labels), go_labels[:10])

# 字段清单用于数据质量检查。
# non_null_count和missing_rate能帮助判断哪些字段适合建模、哪些只能做辅助信息。
main_columns = pd.DataFrame({
    'column_name': main_df.columns,
    'dtype': [str(main_df[c].dtype) for c in main_df.columns],
    'non_null_count': [main_df[c].notna().sum() for c in main_df.columns],
    'missing_count': [main_df[c].isna().sum() for c in main_df.columns],
})
main_columns['missing_rate'] = main_columns['missing_count'] / len(main_df)
display(main_columns)

## 2.展开嵌套comments字段

原始数据把评论和回复压在同一个嵌套字段里，不能直接做评论级建模。本节把每个帖子下的顶层评论展开成评论表，把每条评论下的回复展开成回复表，并尽量保留帖子信息、评论信息和回复信息之间的连接键。展开后的表格是后续清洗、情绪识别和热度指标构造的基础。

In [ ]:
# 本代码块目标：把帖子级嵌套comments字段展开成评论表和回复表。
# 输入：原始帖子DataFrame中的帖子字段、评论字段和回复字段。
# 输出：comments_flat_raw.parquet和replies_flat_raw.parquet。
# 处理思路：逐个帖子读取评论列表，再逐条评论读取回复列表，保留post_id和comment_id连接键。

def dict_of_lists_to_items(d):
    # 某些嵌套对象可能表现为“字典里每个键对应一个列表”。
    # 这个函数把后一种结构拆成逐条记录，避免评论字段错位。
    if not isinstance(d, dict):
        return []
    # 先检查所有列表型字段长度，最长长度决定应该拆出多少条记录。
    lengths = [len(v) for v in d.values() if isinstance(v, (list, tuple, np.ndarray))]
    if not lengths:
        return [d]
    n = max(lengths)
    items = []
    # 逐行重建记录。
    # 如果某个字段列表较短，缺失位置补None，保证每条记录字段数一致。
    for i in range(n):
        item = {}
        for k, v in d.items():
            if isinstance(v, np.ndarray):
                v = v.tolist()
            if isinstance(v, (list, tuple)):
                item[k] = v[i] if i < len(v) else None
            else:
                item[k] = v
        items.append(item)
    return items

def iter_comments(raw_comments):
    # 把原始comments字段统一整理成可遍历的评论字典列表。
    objs = normalize_nested_object(raw_comments)
    out = []
    for obj in objs:
        if isinstance(obj, dict):
            out.extend(dict_of_lists_to_items(obj))
    return out

def iter_replies(raw_replies):
    # 回复字段和评论字段结构类似，也需要统一展开。
    objs = normalize_nested_object(raw_replies)
    out = []
    for obj in objs:
        if isinstance(obj, dict):
            out.extend(dict_of_lists_to_items(obj))
    return out

def flatten_comments_and_replies(df):
    # 这个函数是从帖子级数据走向评论级数据的关键。
    # 输出两张表：顶层评论表和回复表，两者通过post_id与评论索引保持连接。
    comment_rows, reply_rows = [], []
    # 逐个帖子展开。
    # 使用iterrows速度一般，但这里更直观，且只执行一次中间数据构建。
    for _, row in df.iterrows():
        # 先保存帖子级上下文。
        # 这些字段会复制到每条评论，保证评论级分析仍能回到所属帖子和subreddit。
        post = {
            'post_id': row.get('id'),
            'post_title': row.get('post_title'),
            'post_author': row.get('post_author'),
            'post_body': row.get('post_body'),
            'post_url': row.get('post_url'),
            'post_pic': row.get('post_pic'),
            'subreddit': row.get('subreddit'),
            'post_timestamp': row.get('post_timestamp'),
            'post_upvotes': row.get('post_upvotes'),
            'post_permalink': row.get('post_permalink'),
        }
        # ci是同一帖子内的评论序号。
        # 当原始评论没有稳定ID时，post_id+comment_index可以作为临时连接键。
        for ci, c in enumerate(iter_comments(row.get('comments'))):
            # 每条评论下可能有回复，也可能没有。
            # 候选字段兼容不同命名，空回复统一为列表。
            replies = iter_replies(get_first_existing(c, ['replies','Replies','reply','Reply'], []))
            reply_upvotes = [pd.to_numeric(get_first_existing(r, ['ReplyUpvotes','reply_upvotes','upvotes','score'], 0), errors='coerce') for r in replies]
            reply_upvotes_sum = float(np.nansum([0 if pd.isna(x) else x for x in reply_upvotes]))
            # c_row是评论级记录。
            # 除了评论文本，还保留回复数和回复upvotes汇总，后面会进入评论热度指标。
            c_row = {
                **post,
                'comment_index': ci,
                'comment_author': get_first_existing(c, ['CommentAuthor','comment_author','author']),
                'comment_body': get_first_existing(c, ['CommentBody','comment_body','body','text']),
                'comment_timestamp': get_first_existing(c, ['CommentTimestamp','comment_timestamp','timestamp','created_utc','created']),
                'comment_upvotes': get_first_existing(c, ['CommentUpvotes','comment_upvotes','upvotes','score']),
                'comment_permalink': get_first_existing(c, ['CommentPermalink','comment_permalink','permalink']),
                'reply_count': len(replies),
                'reply_upvotes_sum': reply_upvotes_sum,
                'has_replies': len(replies) > 0,
            }
            comment_rows.append(c_row)
            # 回复表单独保存，因为回复文本也会参与情绪识别。
            # ri记录同一评论下的回复序号，便于回溯嵌套位置。
            for ri, r in enumerate(replies):
                reply_rows.append({
                    'post_id': post['post_id'],
                    'post_title': post['post_title'],
                    'subreddit': post['subreddit'],
                    'post_timestamp': post['post_timestamp'],
                    'post_upvotes': post['post_upvotes'],
                    'post_permalink': post['post_permalink'],
                    'parent_comment_index': ci,
                    'parent_comment_author': c_row['comment_author'],
                    'parent_comment_body': c_row['comment_body'],
                    'parent_comment_upvotes': c_row['comment_upvotes'],
                    'reply_index': ri,
                    'reply_author': get_first_existing(r, ['ReplyAuthor','reply_author','author']),
                    'reply_body': get_first_existing(r, ['ReplyBody','reply_body','body','text']),
                    'reply_timestamp': get_first_existing(r, ['ReplyTimestamp','reply_timestamp','timestamp','created_utc','created']),
                    'reply_upvotes': get_first_existing(r, ['ReplyUpvotes','reply_upvotes','upvotes','score']),
                })
    return pd.DataFrame(comment_rows), pd.DataFrame(reply_rows)

# 如果展开结果已经存在，优先读取，避免每次运行都重复解析嵌套结构。
if COMMENTS_FLAT_PATH.exists() and REPLIES_FLAT_PATH.exists():
    comments_flat = pd.read_parquet(COMMENTS_FLAT_PATH)
    replies_flat = pd.read_parquet(REPLIES_FLAT_PATH)
else:
    comments_flat, replies_flat = flatten_comments_and_replies(main_df)
    save_parquet(comments_flat, COMMENTS_FLAT_PATH)
    save_parquet(replies_flat, REPLIES_FLAT_PATH)

print('comments_flat:', comments_flat.shape)
print('replies_flat:', replies_flat.shape)
display(comments_flat.head())

## 3.数据清洗与标准化

本节处理空文本、删除占位符、去除重复文本、整理时间字段和数值字段，并构造文本长度、词数、URL标记、问号标记和感叹号标记。清洗目标是让文本样本更稳定、字段类型更明确、后续模型输入更可控，同时保留评论原始表达。

In [ ]:
# 本代码块目标：清洗评论与回复文本，并标准化时间、互动数和文本特征。
# 输入：展开后的评论表和回复表。
# 输出：comments_clean.parquet和replies_clean.parquet。
# 处理思路：删除无效文本与重复文本，转换数值字段，生成长度、词数和标点特征。

def clean_comment_level(df):
    # 评论清洗以保留真实可分析文本为目标。
    # 每个剔除条件都保留成布尔列，便于后续检查究竟删掉了什么。
    out = df.copy()
    # 保留原始文本列，清洗文本另存，避免覆盖原始内容。
    out['comment_body_raw'] = out.get('comment_body', pd.Series('', index=out.index))
    out['comment_text_clean'] = out['comment_body_raw'].apply(clean_text_basic)
    out['is_missing_text'] = out['comment_body_raw'].isna()
    out['is_empty_text'] = out['comment_text_clean'].eq('')
    out['is_deleted_removed'] = out['comment_text_clean'].apply(is_deleted_or_removed)
    out['is_duplicate_text'] = out['comment_text_clean'].duplicated(keep='first')
    out['is_short_text_under_5'] = out['comment_text_clean'].str.len() < 5
    out['is_us_political'] = out['comment_text_clean'].apply(contains_us_political_keywords)
    # mask代表最终保留条件。
    # 这里采用“只剔除明显不可用文本”的保守策略，避免过度清洗。
    mask = ~(out['is_missing_text'] | out['is_empty_text'] | out['is_deleted_removed'] | out['is_duplicate_text'] | out['is_short_text_under_5'] | out['is_us_political'])
    out = out.loc[mask].copy()
    # subreddit统一小写，避免同一社区因大小写差异被分成多个类别。
    out['subreddit'] = out['subreddit'].astype(str).str.strip().str.lower()
    out['post_timestamp_dt'] = safe_to_datetime(out.get('post_timestamp'))
    out['comment_timestamp_dt'] = safe_to_datetime(out.get('comment_timestamp'))
    # 互动数字统一转为数值。
    # 缺失值填0是因为没有记录到互动时，不应让后续log转换报错。
    for col in ['post_upvotes','comment_upvotes','reply_count','reply_upvotes_sum']:
        if col in out.columns:
            out[col + '_num'] = safe_to_numeric(out[col]).fillna(0)
    # normalize expected names
    out['post_upvotes_num'] = out.get('post_upvotes_num', safe_to_numeric(out.get('post_upvotes', 0)).fillna(0))
    out['comment_upvotes_num'] = out.get('comment_upvotes_num', safe_to_numeric(out.get('comment_upvotes', 0)).fillna(0))
    out['reply_count_num'] = out.get('reply_count_num', safe_to_numeric(out.get('reply_count', 0)).fillna(0))
    out['reply_upvotes_sum_num'] = out.get('reply_upvotes_sum_num', safe_to_numeric(out.get('reply_upvotes_sum', 0)).fillna(0))
    # upvotes等互动指标通常右偏，log1p能压缩极端值影响。
    # clip(lower=0)避免极少数负分评论导致log无定义。
    out['post_upvotes_log'] = np.log1p(out['post_upvotes_num'].clip(lower=0))
    out['comment_upvotes_log'] = np.log1p(out['comment_upvotes_num'].clip(lower=0))
    out['reply_count_log'] = np.log1p(out['reply_count_num'].clip(lower=0))
    out['reply_upvotes_sum_log'] = np.log1p(out['reply_upvotes_sum_num'].clip(lower=0))
    # 拼接文本形式特征，让后续模型能控制评论长度、URL和标点等表达差异。
    out = pd.concat([out, text_features(out['comment_text_clean'])], axis=1)
    return out

def clean_reply_level(df):
    # 回复清洗逻辑与评论一致。
    # 单独写函数是因为回复字段名和评论字段名不同，混在一起反而更难读。
    out = df.copy()
    out['reply_body_raw'] = out.get('reply_body', pd.Series('', index=out.index))
    out['reply_text_clean'] = out['reply_body_raw'].apply(clean_text_basic)
    out['is_missing_text'] = out['reply_body_raw'].isna()
    out['is_empty_text'] = out['reply_text_clean'].eq('')
    out['is_deleted_removed'] = out['reply_text_clean'].apply(is_deleted_or_removed)
    out['is_duplicate_text'] = out['reply_text_clean'].duplicated(keep='first')
    out['is_short_text_under_5'] = out['reply_text_clean'].str.len() < 5
    out['is_us_political'] = out['reply_text_clean'].apply(contains_us_political_keywords)
    mask = ~(out['is_missing_text'] | out['is_empty_text'] | out['is_deleted_removed'] | out['is_duplicate_text'] | out['is_short_text_under_5'] | out['is_us_political'])
    out = out.loc[mask].copy()
    out['subreddit'] = out['subreddit'].astype(str).str.strip().str.lower()
    out['reply_timestamp_dt'] = safe_to_datetime(out.get('reply_timestamp'))
    out['post_upvotes_num'] = safe_to_numeric(out.get('post_upvotes', 0)).fillna(0)
    out['parent_comment_upvotes_num'] = safe_to_numeric(out.get('parent_comment_upvotes', 0)).fillna(0)
    out['reply_upvotes_num'] = safe_to_numeric(out.get('reply_upvotes', 0)).fillna(0)
    out['reply_upvotes_log'] = np.log1p(out['reply_upvotes_num'].clip(lower=0))
    out = pd.concat([out, text_features(out['reply_text_clean'])], axis=1)
    return out

# 清洗结果存在时直接读取，保证后续阶段可以快速恢复。
if COMMENTS_CLEAN_PATH.exists() and REPLIES_CLEAN_PATH.exists():
    comments_clean = pd.read_parquet(COMMENTS_CLEAN_PATH)
    replies_clean = pd.read_parquet(REPLIES_CLEAN_PATH)
else:
    comments_clean = clean_comment_level(comments_flat)
    replies_clean = clean_reply_level(replies_flat)
    save_parquet(comments_clean, COMMENTS_CLEAN_PATH)
    save_parquet(replies_clean, REPLIES_CLEAN_PATH)

print('comments_clean:', comments_clean.shape)
print('replies_clean:', replies_clean.shape)

## 4.GoEmotions情绪识别

本节使用本地RoBERTa-GoEmotions模型识别评论和回复中的情绪概率。GoEmotions包含27类具体情绪和neutral标签，本项目进一步把这些概率整理成正向情绪、负向情绪、高唤醒情绪、情绪效价、情绪强度和主导情绪等变量。由于模型推理耗时较长，如果正式情绪结果文件已存在，notebook会优先读取。

In [ ]:
# 本代码块目标：读取GoEmotions标签、情绪映射和本地模型配置。
# 输入：GoEmotions标签文件、sentiment映射、Ekman映射和本地模型目录。
# 输出：标签列表、正向情绪、负向情绪、高唤醒情绪和模型可用性检查。
# 处理思路：先确认情绪体系完整，再进入耗时的模型推理。

def read_goemotions_mappings():
    # 读取情绪标签和分组。
    # 分组用于把28个细粒度情绪概率合成为正向、负向、高唤醒等分析变量。
    labels = [x.strip() for x in (GOEMOTIONS_DIR / 'emotions.txt').read_text(encoding='utf-8').splitlines() if x.strip()]
    # default_groups是兜底分组。
    # 即使映射JSON缺失，也能用这里的人工分组继续构造情绪变量。
    default_groups = {
        'positive': ['admiration','amusement','approval','caring','desire','excitement','gratitude','joy','love','optimism','pride','relief'],
        'negative': ['anger','annoyance','disappointment','disapproval','disgust','embarrassment','fear','grief','nervousness','remorse','sadness'],
        'ambiguous': ['confusion','curiosity','realization','surprise','neutral'],
        'high_arousal': ['anger','annoyance','disgust','fear','excitement','surprise','nervousness'],
    }
    # 优先使用项目内的sentiment_mapping.json。
    # 如果文件格式异常，再回到default_groups，保证运行链路更稳。
    try:
        sent = json.loads((GOEMOTIONS_DIR / 'sentiment_mapping.json').read_text(encoding='utf-8'))
        positive = sent.get('positive', default_groups['positive'])
        negative = sent.get('negative', default_groups['negative'])
        ambiguous = sent.get('ambiguous', default_groups['ambiguous'])
    except Exception:
        positive, negative, ambiguous = default_groups['positive'], default_groups['negative'], default_groups['ambiguous']
    # 高唤醒情绪单独设置，因为它描述情绪激烈程度，和正负分类维度不同。
    high_arousal = default_groups['high_arousal']
    return labels, positive, negative, ambiguous, high_arousal

label_names_from_file, positive_labels, negative_labels, ambiguous_labels, high_arousal_labels = read_goemotions_mappings()
print('labels from emotions.txt:', len(label_names_from_file))

In [ ]:
# 本代码块目标：完成情绪推理、情绪概率整理和情绪变量构造。
# 输入：清洗后的评论、回复、本地RoBERTa-GoEmotions模型和标签映射。
# 输出：comments_with_emotions.parquet和replies_with_emotions.parquet。
# 处理思路：若正式情绪文件已存在则直接读取；否则分批推理并保存结果。
# 注意事项：CPU推理时间较长，保存中间结果可以避免每次运行都重复计算。

def add_emotion_features(df, label_names, positive_labels, negative_labels, ambiguous_labels, high_arousal_labels):
    # 模型输出是每种情绪的概率，研究需要的是更容易解释的组合变量。
    # 这个函数把细粒度概率转成正向、负向、高唤醒、效价和主导情绪。
    out = df.copy()
    # 只使用实际存在的概率列，避免模型标签和表格列轻微不一致时报错。
    prob_cols = [f'emotion_prob_{x}' for x in label_names if f'emotion_prob_{x}' in out.columns]
    def sum_probs(labels):
        # 同一组情绪的概率相加，得到更高层的情绪维度。
        # 例如negative_score表示多种负向情绪的整体强度。
        cols = [f'emotion_prob_{x}' for x in labels if f'emotion_prob_{x}' in out.columns]
        return out[cols].sum(axis=1) if cols else 0
    out['positive_score'] = sum_probs(positive_labels)
    out['negative_score'] = sum_probs(negative_labels)
    out['ambiguous_score'] = sum_probs(ambiguous_labels)
    out['arousal_score'] = sum_probs(high_arousal_labels)
    # 情绪效价用正向减负向，数值越高表示整体越偏正向。
    out['sentiment_valence'] = out['positive_score'] - out['negative_score']
    out['neutral_score'] = out['emotion_prob_neutral'] if 'emotion_prob_neutral' in out.columns else 0
    # emotion_intensity不把neutral算进去，因为我们关心的是非中性情绪强度。
    non_neutral = [c for c in prob_cols if c != 'emotion_prob_neutral']
    out['emotion_intensity'] = out[non_neutral].max(axis=1) if non_neutral else 0
    if prob_cols:
        idx = out[prob_cols].values.argmax(axis=1)
        out['dominant_emotion'] = [prob_cols[i].replace('emotion_prob_','') for i in idx]
        out['dominant_emotion_score'] = out[prob_cols].max(axis=1)
    if non_neutral:
        idx2 = out[non_neutral].values.argmax(axis=1)
        out['dominant_non_neutral_emotion'] = [non_neutral[i].replace('emotion_prob_','') for i in idx2]
        out['dominant_non_neutral_score'] = out[non_neutral].max(axis=1)
    return out

def predict_goemotions_cpu(texts, tokenizer, model, label_names, batch_size=8, max_length=128):
    # 本函数在CPU上分批推理。
    # 分批的原因是一次性输入全部文本会占用过多内存。
    import torch
    from tqdm.auto import tqdm
    rows = []
    texts = [' ' if clean_text_basic(t) == '' else str(t) for t in texts]
    # 每个batch独立编码、推理、转成概率表。
    # tqdm进度条让长时间运行时能看到当前位置。
    for start in tqdm(range(0, len(texts), batch_size), desc='GoEmotions'):
        batch = texts[start:start+batch_size]
        # truncation限制文本长度，padding保证同一batch能组成张量。
        enc = tokenizer(batch, truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        # 推理阶段不需要梯度，关闭梯度可以降低内存占用。
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.sigmoid(logits).cpu().numpy()
        rows.append(pd.DataFrame(probs, columns=[f'emotion_prob_{x}' for x in label_names]))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=[f'emotion_prob_{x}' for x in label_names])

def run_emotion_if_needed():
    # 只有情绪parquet缺失时才会调用本函数。
    # 这样平时运行notebook可以直接读取已有结果，节省大量时间。
    require_file(MODEL_DIR / 'model.safetensors', 'local roberta model')
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR), local_files_only=True)
    # 统一使用CPU，避免不同电脑是否有GPU导致运行方式不同。
    model.to('cpu')
    model.eval()
    label_names = [model.config.id2label[i] for i in range(len(model.config.id2label))]
    comment_probs = predict_goemotions_cpu(comments_clean['comment_text_clean'].tolist(), tokenizer, model, label_names, BATCH_SIZE, MAX_LENGTH)
    reply_probs = predict_goemotions_cpu(replies_clean['reply_text_clean'].tolist(), tokenizer, model, label_names, BATCH_SIZE, MAX_LENGTH)
    comments_out = add_emotion_features(pd.concat([comments_clean.reset_index(drop=True), comment_probs], axis=1), label_names, positive_labels, negative_labels, ambiguous_labels, high_arousal_labels)
    replies_out = add_emotion_features(pd.concat([replies_clean.reset_index(drop=True), reply_probs], axis=1), label_names, positive_labels, negative_labels, ambiguous_labels, high_arousal_labels)
    save_parquet(comments_out, COMMENTS_EMOTION_PATH)
    save_parquet(replies_out, REPLIES_EMOTION_PATH)
    return comments_out, replies_out

# 情绪结果存在时优先读取，这是最耗时步骤的缓存入口。
if COMMENTS_EMOTION_PATH.exists() and REPLIES_EMOTION_PATH.exists():
    comments_emotions = pd.read_parquet(COMMENTS_EMOTION_PATH)
    replies_emotions = pd.read_parquet(REPLIES_EMOTION_PATH)
elif RUN_EMOTION_IF_MISSING:
    comments_emotions, replies_emotions = run_emotion_if_needed()
else:
    raise FileNotFoundError('情绪识别结果不存在。请启用RUN_EMOTION_IF_MISSING或恢复正式parquet。')

emotion_prob_cols = [c for c in comments_emotions.columns if c.startswith('emotion_prob_')]
label_names = [c.replace('emotion_prob_','') for c in emotion_prob_cols]
print('comments_emotions:', comments_emotions.shape)
print('replies_emotions:', replies_emotions.shape)
print('emotion probability columns:', len(emotion_prob_cols))

## 5.TF-IDF+NMF主题建模

本节使用TF-IDF把评论文本转换为词项权重矩阵，再用NMF识别主要讨论主题。主题建模的作用是为评论内容提供议题结构，使后续模型可以区分“情绪效果”和“议题差异”。主题编号由模型生成，中文和英文主题标签依据关键词和样本文本解释得到。

In [ ]:
# 本代码块目标：用TF-IDF+NMF识别评论主题，并给主题添加可读标签。
# 输入：带情绪变量的评论表。
# 输出：comments_with_topics.parquet和topic_label_mapping.csv。
# 处理思路：先把文本转成词项权重，再用NMF得到主题，再根据关键词解释主题名称。

# 主题编号来自NMF模型，主题名称来自关键词和样本文本的人工解释。
# 同时保存中文和英文标签，是为了兼顾报告正文和英文图表。
TOPIC_LABEL_MAP = {
    -1: ('未分配', 'Unassigned'),
    0: ('气候变化综合讨论与全球影响', 'General Climate Change Discussion and Global Impacts'),
    1: ('油气能源、煤炭与价格争议', 'Oil, Gas, Coal, and Energy Price Debate'),
    2: ('气候历史、冰期与长期变暖尺度', 'Climate History, Ice Ages, and Long-Term Warming'),
    3: ('太阳能、风能、核电与电网转型', 'Solar, Wind, Nuclear Power, and Grid Transition'),
    4: ('塑料污染、零废弃与循环消费', 'Plastic Pollution, Zero Waste, and Circular Consumption'),
    5: ('化石燃料产业、碳排放与能源转型', 'Fossil Fuel Industry, Carbon Emissions, and Energy Transition'),
    6: ('怀疑、否认与消极态度表达', 'Skepticism, Denial, and Negative Personal Attitudes'),
    7: ('肉类消费、素食主义与饮食转型', 'Meat Consumption, Veganism, and Dietary Transition'),
    8: ('行动呼吁、电动车与基础设施建设', 'Calls for Action, EVs, and Infrastructure Development'),
    9: ('好消息、积极评价与解决方案想象', 'Good News, Positive Evaluation, and Solution Framing'),
}

def preprocess_for_topic_modeling(text):
    # 主题建模前的文本处理比情绪识别更激进。
    # 这里移除URL、subreddit标记和非英文字符，让主题词更集中。
    t = clean_text_basic(text).lower()
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'\b[ur]/[a-zA-Z0-9_]+', ' ', t)
    t = re.sub(r'&[a-z]+;', ' ', t)
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def run_topic_modeling_if_needed(comments_df):
    # 主题模型只在comments_with_topics.parquet缺失时运行。
    # 已有主题结果时直接读取，避免主题编号因为重新训练而产生不必要变化。
    from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
    from sklearn.decomposition import NMF
    out = comments_df.copy()
    # topic_text是专门给主题模型使用的文本列，不覆盖原始清洗文本。
    out['topic_text'] = out['comment_text_clean'].apply(preprocess_for_topic_modeling)
    wc = out['topic_text'].str.split().str.len().fillna(0)
    # 过短文本很难提供稳定主题信息，因此不参与NMF训练。
    # 这些评论保留在数据里，但topic_id设为-1。
    eligible = out['topic_text'].ne('') & (wc >= 5) & (out['topic_text'].str.len() >= 20)
    stop = set(ENGLISH_STOP_WORDS) | {'climate','change','energy','people','just','like','think','know','make','use','time','really','would','could','also','one','get','much','many','thing','things','way','going','say','said','reddit','comment','post'}
    # TF-IDF把文本转成词项权重矩阵。
    # min_df过滤极低频词，max_df过滤几乎所有评论都会出现的泛化词。
    vectorizer = TfidfVectorizer(stop_words=list(stop), max_df=0.85, min_df=10, max_features=8000, ngram_range=(1,2), token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')
    X = vectorizer.fit_transform(out.loc[eligible, 'topic_text'])
    # NMF把词项矩阵分解为10个非负主题。
    # nndsvda初始化通常比随机初始化更稳定。
    nmf = NMF(n_components=10, random_state=RANDOM_STATE, init='nndsvda', max_iter=500)
    W = nmf.fit_transform(X)
    row_sum = W.sum(axis=1)
    topic_id = W.argmax(axis=1)
    topic_score = W.max(axis=1)
    # topic_score_normalized表示主导主题在该评论主题权重中的相对占比。
    # where=row_sum!=0避免空行除以0。
    topic_score_norm = np.divide(topic_score, row_sum, out=np.zeros_like(topic_score), where=row_sum!=0)
    out['topic_id'] = -1
    out['topic_score'] = 0.0
    out['topic_score_normalized'] = 0.0
    out.loc[eligible, 'topic_id'] = topic_id
    out.loc[eligible, 'topic_score'] = topic_score
    out.loc[eligible, 'topic_score_normalized'] = topic_score_norm
    out['topic_label'] = out['topic_id'].map(lambda x: f'Topic {x}' if x >= 0 else 'Unassigned')
    out['topic_label_zh'] = out['topic_id'].map(lambda x: TOPIC_LABEL_MAP.get(int(x), ('未命名','Unnamed'))[0])
    out['topic_label_en'] = out['topic_id'].map(lambda x: TOPIC_LABEL_MAP.get(int(x), ('未命名','Unnamed'))[1])
    save_parquet(out, COMMENTS_TOPIC_PATH)
    return out

# 已有主题结果时直接读取，保持主题编号和标签稳定。
if COMMENTS_TOPIC_PATH.exists():
    comments_topics = pd.read_parquet(COMMENTS_TOPIC_PATH)
else:
    comments_topics = run_topic_modeling_if_needed(comments_emotions)

if 'topic_label_zh' not in comments_topics.columns:
    comments_topics['topic_label_zh'] = comments_topics['topic_id'].map(lambda x: TOPIC_LABEL_MAP.get(int(x), ('未命名','Unnamed'))[0])
if 'topic_label_en' not in comments_topics.columns:
    comments_topics['topic_label_en'] = comments_topics['topic_id'].map(lambda x: TOPIC_LABEL_MAP.get(int(x), ('未命名','Unnamed'))[1])

print('comments_topics:', comments_topics.shape)
display(comments_topics[['comment_text_clean','topic_id','topic_label_en','dominant_emotion']].head())

## 6.帖子层情感极化指标

本节把评论级情绪结果聚合到帖子层，构造情感极化和情绪结构变量。重点变量包括情绪分布差异、情绪效价标准差、主导情绪多样性、正负情绪并存程度、高唤醒评论占比和负向评论占比。它们用于描述同一帖子下评论情绪是否分化、冲突或多样化。

In [ ]:
# 本代码块目标：把评论层情绪聚合到帖子层，并计算情感极化指标。
# 输入：带主题和情绪的评论表、带情绪的回复表、原始帖子表。
# 输出：post_level_polarization.parquet。
# 处理思路：按post_id聚合情绪均值、占比、熵、分布差异和回复变化指标。

def entropy_norm(values):
    # 熵用于描述一个帖子内主导情绪或主题是否多样。
    # 标准化后范围更容易比较，0表示几乎没有多样性。
    counts = pd.Series(values).dropna().value_counts()
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum() / np.log(len(counts)))

def mean_pairwise_jsd(matrix, max_n=300):
    # Jensen-Shannon距离用于衡量评论情绪分布之间差异。
    # max_n限制最多比较300条评论，避免长帖组合数量过大。
    from scipy.spatial.distance import jensenshannon
    arr = np.asarray(matrix, dtype=float)
    if arr.shape[0] < 2:
        return np.nan
    # 长帖评论数过多时做固定随机抽样。
    # 这保留估计可行性，也防止单个帖子拖慢整本notebook。
    if arr.shape[0] > max_n:
        rng = np.random.default_rng(RANDOM_STATE)
        arr = arr[rng.choice(arr.shape[0], size=max_n, replace=False)]
    arr = np.nan_to_num(arr, nan=0.0)
    sums = arr.sum(axis=1, keepdims=True)
    # 每条评论的情绪概率先归一化为分布。
    # 如果某行概率和为0，则用均匀分布兜底。
    arr = np.divide(arr, sums, out=np.full_like(arr, 1/arr.shape[1]), where=sums!=0)
    vals = []
    for i, j in combinations(range(arr.shape[0]), 2):
        vals.append(jensenshannon(arr[i], arr[j]))
    return float(np.mean(vals)) if vals else np.nan

def build_post_polarization(comments, replies):
    # 这是从评论层走向帖子层的关键函数。
    # 每个post_id聚合成一行，保存该帖子内部情绪结构和互动热度。
    comments = comments.copy()
    # 高唤醒阈值使用全体评论75分位数。
    # 这样high_arousal_share表示帖子内有多少评论超过整体较高唤醒水平。
    arousal_q75 = comments['arousal_score'].quantile(0.75)
    prob_cols = [c for c in comments.columns if c.startswith('emotion_prob_')]
    rows = []
    # 按帖子逐组聚合评论。
    # g包含同一帖子下全部顶层评论，是计算帖子内部情感极化的基本单位。
    for post_id, g in comments.groupby('post_id', dropna=False):
        first = g.iloc[0]
        topic_counts = g['topic_id'].value_counts(dropna=False)
        # 主导主题取该帖子下评论最多的主题。
        # 如果没有主题信息，则用-1表示未分配。
        dominant_topic_id = int(topic_counts.index[0]) if len(topic_counts) else -1
        # negative_share和positive_share表示评论立场倾向占比。
        # 它们用于构造“正负情绪并存”的冲突指标。
        neg_share = float((g['negative_score'] > g['positive_score']).mean())
        pos_share = float((g['positive_score'] > g['negative_score']).mean())
        emo_entropy = entropy_norm(g['dominant_emotion'])
        rows.append({
            'post_id': post_id,
            'post_title': first.get('post_title'),
            'subreddit': first.get('subreddit'),
            'post_timestamp': first.get('post_timestamp'),
            'post_upvotes_num': pd.to_numeric(first.get('post_upvotes_num', 0), errors='coerce'),
            'post_comment_count': len(g),
            'post_reply_count_sum': g['reply_count_num'].sum(),
            'post_reply_upvotes_sum': g['reply_upvotes_sum_num'].sum(),
            'post_comment_upvotes_sum': g['comment_upvotes_num'].sum(),
            'post_comment_upvotes_mean': g['comment_upvotes_num'].mean(),
            'dominant_topic_id': dominant_topic_id,
            'dominant_topic_label': TOPIC_LABEL_MAP.get(dominant_topic_id, ('未命名','Unnamed'))[1],
            'dominant_topic_label_zh': TOPIC_LABEL_MAP.get(dominant_topic_id, ('未命名','Unnamed'))[0],
            'dominant_topic_label_en': TOPIC_LABEL_MAP.get(dominant_topic_id, ('未命名','Unnamed'))[1],
            'topic_entropy': entropy_norm(g['topic_id']),
            'topic_count': g['topic_id'].nunique(),
            'mean_positive_score': g['positive_score'].mean(),
            'mean_negative_score': g['negative_score'].mean(),
            'mean_arousal_score': g['arousal_score'].mean(),
            'mean_sentiment_valence': g['sentiment_valence'].mean(),
            'mean_emotion_intensity': g['emotion_intensity'].mean(),
            'mean_neutral_score': g.get('neutral_score', pd.Series(0,index=g.index)).mean(),
            'std_positive_score': g['positive_score'].std(),
            'std_negative_score': g['negative_score'].std(),
            'std_arousal_score': g['arousal_score'].std(),
            'std_sentiment_valence': g['sentiment_valence'].std(),
            'std_emotion_intensity': g['emotion_intensity'].std(),
            'emotion_jsd_polarization': mean_pairwise_jsd(g[prob_cols].to_numpy()) if prob_cols else np.nan,
            'valence_std_polarization': g['sentiment_valence'].std(),
            'pos_neg_balance_gap': abs(g['positive_score'].mean() - g['negative_score'].mean()),
            'negative_share': neg_share,
            'positive_share': pos_share,
            'high_arousal_share': float((g['arousal_score'] > arousal_q75).mean()),
            'dominant_emotion_entropy': emo_entropy,
            # emotion_conflict_index同时要求负向评论占比、正向评论占比和情绪多样性都较高。
            # 如果只有单边情绪，或者情绪类型很单一，冲突指数都会降低。
            'emotion_conflict_index': neg_share * pos_share * emo_entropy,
        })
    posts = pd.DataFrame(rows)
    # 帖子互动字段统一数值化，避免后续热度计算混入字符串或空值。
    for col in ['post_upvotes_num','post_comment_count','post_reply_count_sum','post_comment_upvotes_sum','post_reply_upvotes_sum']:
        posts[col] = pd.to_numeric(posts[col], errors='coerce').fillna(0)
    # post_heat_raw合并帖子upvotes、评论数和回复数，表示帖子内部互动规模。
    posts['post_heat_raw'] = posts['post_upvotes_num'].clip(lower=0) + posts['post_comment_count'] + posts['post_reply_count_sum']
    posts['post_heat_log'] = np.log1p(posts['post_heat_raw'])
    posts['comment_discussion_heat'] = np.log1p(posts['post_comment_count']) + np.log1p(posts['post_reply_count_sum'])
    posts['engagement_heat'] = np.log1p(posts['post_upvotes_num'].clip(lower=0)) + np.log1p(posts['post_comment_upvotes_sum'].clip(lower=0)) + np.log1p(posts['post_reply_upvotes_sum'].clip(lower=0))
    return posts

# 帖子层极化结果存在时直接读取，减少重复聚合和JSD计算。
if POST_POLARIZATION_PATH.exists():
    post_polar = pd.read_parquet(POST_POLARIZATION_PATH)
else:
    post_polar = build_post_polarization(comments_topics, replies_emotions)
    save_parquet(post_polar, POST_POLARIZATION_PATH)

print('post_polar:', post_polar.shape)
display(post_polar.head())

## 7.传播热度指标与最终建模数据集

本节把评论互动和帖子互动整理成可建模的传播热度指标。评论层关注评论upvotes和回复数量，帖子层关注帖子upvotes、评论数、回复upvotes和综合互动。最终会保存评论层建模数据和帖子层建模数据，后续描述统计、统计建模和机器学习都以这两张表为主要输入。

In [ ]:
# 本代码块目标：构造评论层和帖子层传播热度指标，并保存最终建模数据。
# 输入：评论层主题情绪数据和帖子层极化数据。
# 输出：modeling_dataset_comment_level.parquet和modeling_dataset_post_level.parquet。
# 处理思路：对偏态互动数做log转换，并构造top20%高热度分类标签。

def make_modeling_datasets(comments, posts):
    # 最终建模数据分为评论层和帖子层。
    # 评论层回答单条评论热度问题，帖子层回答帖子整体热度和极化问题。
    # 先复制评论表，避免在原始主题情绪表上直接覆盖字段。
    c = comments.copy()
    # 评论层互动字段统一转数值并生成非负版本。
    # 非负版本用于log转换，原始数值仍保留给需要原尺度解释的地方。
    for col in ['post_upvotes_num','comment_upvotes_num','reply_count_num','reply_upvotes_sum_num']:
        c[col] = pd.to_numeric(c.get(col, 0), errors='coerce').fillna(0)
        c[col.replace('_num','_nonneg')] = c[col].clip(lower=0)
    c['comment_upvotes_log'] = np.log1p(c['comment_upvotes_nonneg'])
    c['reply_count_log'] = np.log1p(c['reply_count_nonneg'])
    c['reply_upvotes_sum_log'] = np.log1p(c['reply_upvotes_sum_nonneg'])
    c['comment_heat_simple'] = c['comment_upvotes_log']
    c['comment_discussion_heat'] = c['reply_count_log'] + c['reply_upvotes_sum_log']
    # comment_heat_composite把评论获赞、回复数量和回复获赞合成一个互动热度指标。
    # 回复相关指标权重较低，是为了避免回复多但原评论互动弱的情况被过度放大。
    c['comment_heat_composite'] = c['comment_upvotes_log'] + 0.5*c['reply_count_log'] + 0.3*c['reply_upvotes_sum_log']
    # top20标签把连续热度转成二分类任务，供机器学习模型预测。
    # 阈值使用样本内80分位数，表示相对高热度而非绝对传播量。
    c['high_heat_comment_top20'] = c['comment_heat_composite'] >= c['comment_heat_composite'].quantile(0.8)
    c['high_upvote_comment_top20'] = c['comment_upvotes_num'] >= c['comment_upvotes_num'].quantile(0.8)
    if 'topic_label_zh' not in c.columns:
        c['topic_label_zh'] = c['topic_id'].map(lambda x: TOPIC_LABEL_MAP.get(int(x), ('未命名','Unnamed'))[0])
    if 'topic_label_en' not in c.columns:
        c['topic_label_en'] = c['topic_id'].map(lambda x: TOPIC_LABEL_MAP.get(int(x), ('未命名','Unnamed'))[1])
    # 把帖子层极化变量合并回评论层。
    # 这样评论层模型既能使用单条评论情绪，也能控制所属帖子整体情绪结构。
    post_cols = ['post_id','emotion_jsd_polarization','valence_std_polarization','pos_neg_balance_gap','negative_share','positive_share','high_arousal_share','dominant_emotion_entropy','emotion_conflict_index','post_heat_log','comment_discussion_heat','engagement_heat','dominant_topic_id','dominant_topic_label_zh','dominant_topic_label_en']
    psmall = posts[[x for x in post_cols if x in posts.columns]].rename(columns={'post_heat_log':'post_level_heat_log','comment_discussion_heat':'post_level_discussion_heat','engagement_heat':'post_level_engagement_heat'})
    c = c.merge(psmall, on='post_id', how='left')

    # 帖子层数据单独复制并构造帖子热度标签。
    p = posts.copy()
    # 帖子层互动指标同样采用log1p，减少极端热门帖子对模型的支配。
    p['post_upvotes_log'] = np.log1p(pd.to_numeric(p['post_upvotes_num'], errors='coerce').fillna(0).clip(lower=0))
    p['post_comment_count_log'] = np.log1p(pd.to_numeric(p['post_comment_count'], errors='coerce').fillna(0).clip(lower=0))
    p['post_reply_count_sum_log'] = np.log1p(pd.to_numeric(p['post_reply_count_sum'], errors='coerce').fillna(0).clip(lower=0))
    p['post_comment_upvotes_sum_log'] = np.log1p(pd.to_numeric(p['post_comment_upvotes_sum'], errors='coerce').fillna(0).clip(lower=0))
    p['post_reply_upvotes_sum_log'] = np.log1p(pd.to_numeric(p['post_reply_upvotes_sum'], errors='coerce').fillna(0).clip(lower=0))
    # 帖子层也构造top20高热度标签，用于和评论层预测任务对应。
    p['high_heat_post_top20'] = p['post_heat_log'] >= p['post_heat_log'].quantile(0.8)
    p['high_engagement_post_top20'] = p['engagement_heat'] >= p['engagement_heat'].quantile(0.8)
    p['high_discussion_post_top20'] = p['comment_discussion_heat'] >= p['comment_discussion_heat'].quantile(0.8)
    return c, p

# 如果最终建模表已经存在，直接读取。
# 这让读者可以跳过前面耗时的推理和聚合，快速查看模型和图表逻辑。
if COMMENT_MODEL_PATH.exists() and POST_MODEL_PATH.exists():
    comment_model = pd.read_parquet(COMMENT_MODEL_PATH)
    post_model = pd.read_parquet(POST_MODEL_PATH)
else:
    comment_model, post_model = make_modeling_datasets(comments_topics, post_polar)
    save_parquet(comment_model, COMMENT_MODEL_PATH)
    save_parquet(post_model, POST_MODEL_PATH)

print('comment_model:', comment_model.shape)
print('post_model:', post_model.shape)

## 8.描述统计表生成

本节只生成描述统计表，不生成正式图表。这样安排是为了保证逻辑顺序清楚：先把样本规模、社区结构、主题结构、情绪变量和帖子层极化变量整理成可复核的CSV；等统计建模、机器学习训练和稳健性检验全部完成后，再在最后统一生成图表。

描述统计表回答“数据长什么样”，后续模型回答“变量之间有什么关系”和“哪些特征更能预测高热度”。把这两类任务分开，可以避免还没完成分析就提前画结论图。

In [ ]:
# 本代码块目标：生成描述统计CSV，不在这里绘制正式图表。
# 输入：第7节生成的comment_model和post_model两张建模数据表。
# 输出：社区、主题、情绪、极化和相关矩阵等描述统计CSV。
# 处理思路：先把所有需要复核的基础统计量落到表格，再把图表统一放到最后一节。
# -----------------------------------------------------------------------------
# 8.描述统计表生成
# -----------------------------------------------------------------------------


def descriptive_outputs(comment_df: pd.DataFrame, post_df: pd.DataFrame) -> None:
    # 进入函数后第一步先明确两张表各自承担的角色。
    # comment_df是一行一条评论，适合计算评论数量、评论热度和评论情绪均值。
    # post_df是一行一个帖子，适合计算帖子热度、帖子内部情感冲突和帖子层极化变量。
    # 这些统计表后面会被报告、图表和人工检查共同使用，所以全部保存为CSV。
    """生成描述统计表。"""

    # subreddit_comment按社区聚合评论层数据。
    # comment_count告诉我们样本主要来自哪些社区。
    # mean和median同时保留，是因为互动数据通常右偏，均值和中位数可能差异明显。
    # top_dominant_emotion用于快速观察某个社区最常见的主导情绪标签。
    subreddit_comment = (
        comment_df.groupby('subreddit', dropna=False)
        .agg(
            comment_count=('post_id', 'size'),
            mean_comment_heat_composite=('comment_heat_composite', 'mean'),
            median_comment_heat_composite=('comment_heat_composite', 'median'),
            mean_positive_score=('positive_score', 'mean'),
            mean_negative_score=('negative_score', 'mean'),
            mean_arousal_score=('arousal_score', 'mean'),
            mean_sentiment_valence=('sentiment_valence', 'mean'),
            top_dominant_emotion=('dominant_emotion', lambda s: s.value_counts().index[0] if len(s.dropna()) else np.nan),
        )
        .reset_index()
        .sort_values('comment_count', ascending=False)
    )
    # comment_share把原始计数转成占比，便于判断某个subreddit是否主导样本。
    subreddit_comment['comment_share'] = subreddit_comment['comment_count'] / len(comment_df)
    save_csv(subreddit_comment, OUTPUT_TABLE_DIR / 'descriptive_subreddit_comment_summary.csv')

    # topic_comment按主题聚合评论层数据。
    # 这里同时保留topic_id、中文标签和英文标签，方便报告写中文、图表写英文。
    # mean_emotion_intensity帮助判断某些主题是否更容易出现单一强情绪。
    topic_comment = (
        comment_df.groupby(['topic_id', 'topic_label_zh', 'topic_label_en'], dropna=False)
        .agg(
            comment_count=('post_id', 'size'),
            mean_comment_heat_composite=('comment_heat_composite', 'mean'),
            median_comment_heat_composite=('comment_heat_composite', 'median'),
            mean_positive_score=('positive_score', 'mean'),
            mean_negative_score=('negative_score', 'mean'),
            mean_arousal_score=('arousal_score', 'mean'),
            mean_sentiment_valence=('sentiment_valence', 'mean'),
            mean_emotion_intensity=('emotion_intensity', 'mean'),
            top_dominant_emotion=('dominant_emotion', lambda s: s.value_counts().index[0] if len(s.dropna()) else np.nan),
        )
        .reset_index()
        .sort_values('comment_count', ascending=False)
    )
    # comment_share说明每个主题在评论样本中的占比。
    topic_comment['comment_share'] = topic_comment['comment_count'] / len(comment_df)
    save_csv(topic_comment, OUTPUT_TABLE_DIR / 'descriptive_topic_comment_summary.csv')

    # subreddit_post按社区聚合帖子层数据。
    # 这张表和评论层社区表不同：它关注帖子数量、帖子热度和帖子内部极化结构。
    # 如果某个社区评论很多但帖子热度不高，两个表可以帮助发现这种差异。
    subreddit_post = (
        post_df.groupby('subreddit', dropna=False)
        .agg(
            post_count=('post_id', 'size'),
            mean_post_heat_log=('post_heat_log', 'mean'),
            mean_engagement_heat=('engagement_heat', 'mean'),
            mean_emotion_conflict_index=('emotion_conflict_index', 'mean'),
            mean_high_arousal_share=('high_arousal_share', 'mean'),
            mean_negative_share=('negative_share', 'mean'),
        )
        .reset_index()
        .sort_values('post_count', ascending=False)
    )
    save_csv(subreddit_post, OUTPUT_TABLE_DIR / 'descriptive_subreddit_post_summary.csv')

    # topic_post按帖子主导主题聚合帖子层数据。
    # post_count用于判断该主题下有多少帖子。
    # mean_post_heat_log和mean_engagement_heat用于观察主题热度。
    # mean_emotion_conflict_index和mean_dominant_emotion_entropy用于观察主题下评论情绪是否更分化。
    topic_post = (
        post_df.groupby(['dominant_topic_id', 'dominant_topic_label_zh', 'dominant_topic_label_en'], dropna=False)
        .agg(
            post_count=('post_id', 'size'),
            mean_post_heat_log=('post_heat_log', 'mean'),
            mean_engagement_heat=('engagement_heat', 'mean'),
            mean_emotion_conflict_index=('emotion_conflict_index', 'mean'),
            mean_dominant_emotion_entropy=('dominant_emotion_entropy', 'mean'),
        )
        .reset_index()
        .sort_values('post_count', ascending=False)
    )
    save_csv(topic_post, OUTPUT_TABLE_DIR / 'descriptive_topic_post_summary.csv')

    # emotion_cols是评论层情绪变量的最小说明集合。
    # describe会生成均值、标准差、四分位数等基础统计量。
    # 这张表适合检查情绪概率是否在合理范围内，以及变量是否明显偏态。
    emotion_cols = ['positive_score', 'negative_score', 'arousal_score', 'sentiment_valence', 'emotion_intensity', 'neutral_score']
    emotion_summary = comment_df[emotion_cols].describe(percentiles=[.25, .5, .75]).T.reset_index(names='variable')
    save_csv(emotion_summary, OUTPUT_TABLE_DIR / 'descriptive_emotion_score_summary.csv')

    # pol_cols是帖子层极化、情绪结构和热度变量。
    # 由于部分变量可能在不同运行阶段缺失，先用列表推导保留实际存在的列。
    # 这样即使未来删减变量，描述统计也不会因为某个非必要列缺失而中断。
    pol_cols = [
        'emotion_jsd_polarization', 'valence_std_polarization', 'dominant_emotion_entropy',
        'emotion_conflict_index', 'high_arousal_share', 'negative_share', 'positive_share',
        'post_heat_log', 'engagement_heat'
    ]
    existing_pol_cols = [c for c in pol_cols if c in post_df.columns]
    polarization_summary = post_df[existing_pol_cols].describe(percentiles=[.25, .5, .75]).T.reset_index(names='variable')
    save_csv(polarization_summary, OUTPUT_TABLE_DIR / 'descriptive_polarization_summary.csv')

    # corr_cols用于初步检查帖子层主要变量之间是否高度相关。
    # 相关矩阵用于帮助判断变量关系和潜在共线性。
    # 后面的正式系数解释仍以第9节统计模型为准。
    corr_cols = [
        'post_heat_log', 'engagement_heat', 'comment_discussion_heat', 'emotion_conflict_index',
        'emotion_jsd_polarization', 'dominant_emotion_entropy', 'high_arousal_share',
        'negative_share', 'mean_negative_score', 'mean_arousal_score'
    ]
    corr_cols = [c for c in corr_cols if c in post_df.columns]
    save_csv(post_df[corr_cols].corr().reset_index(names='variable'), OUTPUT_TABLE_DIR / 'descriptive_post_correlation_matrix.csv')

    # 用display展示最常用的三张表，方便读者运行到本节时立即看到样本结构。
    # 展示不替代CSV保存；CSV才是后续报告和复核使用的稳定输出。
    display(subreddit_comment.head(8))
    display(topic_comment.head(10))
    display(polarization_summary)


# 执行描述统计表生成。
# 这里传入第7节保存并读取的两张建模表，保证第8节不依赖额外隐藏数据。
descriptive_outputs(comment_model, post_model)

## 9.统计建模

本节使用评论层和帖子层回归模型检验情绪变量、极化变量与传播热度之间的相关关系。评论层模型关注单条评论的负向情绪、高唤醒情绪、情绪效价和情绪强度；帖子层模型关注情绪冲突、情绪分布差异、高唤醒占比和负向占比。模型中加入subreddit和主题控制变量，以降低社区文化和议题差异带来的混淆。

In [ ]:
# 本代码块目标：拟合评论层和帖子层统计模型，并导出模型结果表。
# 输入：评论层建模数据、帖子层建模数据和第8节保存图表的函数。
# 输出：完整系数表、主要变量表、模型拟合摘要、APA表和系数图表。
# 处理思路：使用OLS解释连续热度，用Logit解释高热度分类，并控制subreddit和主题。

# -----------------------------------------------------------------------------
# 9. 统计建模
# -----------------------------------------------------------------------------
# 评论层：检验负向情绪、高唤醒情绪等变量与评论热度的关系。
# 帖子层：检验情感冲突、情绪多样性等变量与帖子热度的关系。


def add_z_scores(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # 回归中同时放入多种情绪变量，它们量纲相近但分布不同。
    # 标准化后，一个系数可以理解为“自变量提高一个标准差时，因变量如何变化”。
    # 这让负向情绪、高唤醒情绪和情绪冲突指数之间更容易比较。
    """为模型变量添加z-score标准化列。"""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            continue
        values = pd.to_numeric(out[col], errors='coerce')
        std = values.std(ddof=0)
        out[f'{col}_z'] = (values - values.mean()) / std if std and not np.isnan(std) else 0.0
    return out


def fit_statsmodels_if_available(comment_df: pd.DataFrame, post_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # 统计模型承担解释任务：关注变量方向、大小和显著性。
    # 评论层模型回答“单条评论的情绪是否和互动更高有关”。
    # 帖子层模型回答“同一帖子内部情绪冲突是否和帖子整体热度有关”。
    # C(subreddit)和C(topic_id)用于控制社区和主题差异，避免把社区文化误解释成情绪作用。
    """训练统计模型并返回完整系数表、主要结果表和拟合摘要。"""
    import statsmodels.formula.api as smf

    comment_df = add_z_scores(comment_df, ['negative_score', 'arousal_score', 'sentiment_valence', 'emotion_intensity'])
    post_df = add_z_scores(post_df, [
        'emotion_conflict_index', 'emotion_jsd_polarization', 'high_arousal_share', 'negative_share',
        'dominant_emotion_entropy', 'mean_negative_score', 'mean_arousal_score', 'mean_sentiment_valence'
    ])

    model_specs = [
        ('C1', 'comment', 'ols', 'comment_heat_composite', 'comment_heat_composite ~ negative_score_z + arousal_score_z + sentiment_valence_z + emotion_intensity_z + C(subreddit) + C(topic_id)'),
        ('C2', 'comment', 'ols', 'comment_upvotes_log', 'comment_upvotes_log ~ negative_score_z + arousal_score_z + sentiment_valence_z + emotion_intensity_z + C(subreddit) + C(topic_id)'),
        ('C3', 'comment', 'ols', 'reply_count_log', 'reply_count_log ~ negative_score_z + arousal_score_z + sentiment_valence_z + emotion_intensity_z + C(subreddit) + C(topic_id)'),
        ('C4', 'comment', 'logit', 'high_heat_comment_top20', 'high_heat_comment_top20 ~ negative_score_z + arousal_score_z + sentiment_valence_z + emotion_intensity_z + C(subreddit) + C(topic_id)'),
        ('P1', 'post', 'ols', 'post_heat_log', 'post_heat_log ~ emotion_conflict_index_z + emotion_jsd_polarization_z + high_arousal_share_z + negative_share_z + dominant_emotion_entropy_z + mean_negative_score_z + mean_arousal_score_z + mean_sentiment_valence_z + C(subreddit) + C(dominant_topic_id)'),
        ('P2', 'post', 'ols', 'engagement_heat', 'engagement_heat ~ emotion_conflict_index_z + emotion_jsd_polarization_z + high_arousal_share_z + negative_share_z + dominant_emotion_entropy_z + mean_negative_score_z + mean_arousal_score_z + mean_sentiment_valence_z + C(subreddit) + C(dominant_topic_id)'),
        ('P3', 'post', 'ols', 'comment_discussion_heat', 'comment_discussion_heat ~ emotion_conflict_index_z + emotion_jsd_polarization_z + high_arousal_share_z + negative_share_z + dominant_emotion_entropy_z + mean_negative_score_z + mean_arousal_score_z + mean_sentiment_valence_z + C(subreddit) + C(dominant_topic_id)'),
        ('P4', 'post', 'logit', 'high_heat_post_top20', 'high_heat_post_top20 ~ emotion_conflict_index_z + emotion_jsd_polarization_z + high_arousal_share_z + negative_share_z + dominant_emotion_entropy_z + mean_negative_score_z + mean_arousal_score_z + mean_sentiment_valence_z + C(subreddit) + C(dominant_topic_id)'),
    ]

    def prepare_model_data(data: pd.DataFrame, outcome: str, model_type: str) -> pd.DataFrame:
        # statsmodels的公式接口依赖patsy解析变量类型。
        # 如果二分类因变量仍是bool，patsy可能把False/True当成两个类别，进而生成两列endog。
        # logistic回归只需要一列0/1目标，所以这里在拟合前做显式转换。
        # 使用copy可以保护上游数据集，避免为了某个模型而悄悄改变后续分析使用的数据。
        model_data = data.copy()
        if model_type == 'logit':
            if pd.api.types.is_bool_dtype(model_data[outcome]):
                model_data[outcome] = model_data[outcome].astype(int)
            else:
                model_data[outcome] = pd.to_numeric(model_data[outcome], errors='coerce')
        return model_data

    coef_rows = []
    fit_rows = []
    for model_name, level, model_type, outcome, formula in model_specs:
        data = comment_df if level == 'comment' else post_df
        data = prepare_model_data(data, outcome, model_type)
        try:
            if model_type == 'ols':
                result = smf.ols(formula, data=data).fit()
            else:
                result = smf.logit(formula, data=data).fit(disp=False, maxiter=200)
        except Exception as model_exc:
            # 单个模型失败时保留诊断信息并继续执行其他模型。
            # 这样报告可以明确指出哪一项失败，避免整节统计分析全部中断。
            warnings.warn(f'{model_name}拟合失败：{model_exc}')
            fit_rows.append({
                'model_name': model_name,
                'level': level,
                'outcome': outcome,
                'nobs': 0.0,
                'r_squared': np.nan,
                'pseudo_r_squared': np.nan,
                'aic': np.nan,
                'bic': np.nan,
                'backend': f'statsmodels {model_type}',
                'error': str(model_exc),
            })
            continue

        fit_rows.append({
            'model_name': model_name,
            'level': level,
            'outcome': outcome,
            'nobs': float(result.nobs),
            'r_squared': getattr(result, 'rsquared', np.nan),
            'pseudo_r_squared': getattr(result, 'prsquared', np.nan),
            'aic': result.aic,
            'bic': result.bic,
            'backend': f'statsmodels {model_type}',
        })
        conf = result.conf_int()
        for term, coef in result.params.items():
            p_value = result.pvalues[term]
            coef_rows.append({
                'model_name': model_name,
                'level': level,
                'outcome': outcome,
                'term': term,
                'coefficient': coef,
                'std_error': result.bse[term],
                'p_value': p_value,
                'ci_low': conf.loc[term, 0],
                'ci_high': conf.loc[term, 1],
                'nobs': float(result.nobs),
                'r_squared': getattr(result, 'rsquared', np.nan),
                'pseudo_r_squared': getattr(result, 'prsquared', np.nan),
                'aic': result.aic,
                'bic': result.bic,
            })

    if not coef_rows:
        raise RuntimeError('所有统计模型都未完成，请检查变量类型、样本量和公式设定。')

    coef_df = pd.DataFrame(coef_rows)
    coef_df['significance'] = pd.cut(
        coef_df['p_value'],
        bins=[-np.inf, .001, .01, .05, np.inf],
        labels=['***', '**', '*', 'n.s.']
    ).astype(str)
    coef_df['interpretation_direction'] = np.where(coef_df['coefficient'] >= 0, 'positive association', 'negative association')

    selected_terms = [
        'negative_score_z', 'arousal_score_z', 'sentiment_valence_z', 'emotion_intensity_z',
        'emotion_conflict_index_z', 'emotion_jsd_polarization_z', 'high_arousal_share_z',
        'negative_share_z', 'dominant_emotion_entropy_z', 'mean_negative_score_z',
        'mean_arousal_score_z', 'mean_sentiment_valence_z'
    ]
    selected_df = coef_df[coef_df['term'].isin(selected_terms)].copy()
    fit_df = pd.DataFrame(fit_rows)
    return coef_df, selected_df, fit_df


try:
    coef_df, selected_effects, fit_summary = fit_statsmodels_if_available(comment_model, post_model)
    save_csv(coef_df, OUTPUT_TABLE_DIR / 'statistical_model_coefficients.csv')
    save_csv(selected_effects, OUTPUT_TABLE_DIR / 'statistical_selected_effects_summary.csv')
    save_csv(fit_summary, OUTPUT_TABLE_DIR / 'statistical_model_fit_summary.csv')
    save_csv(selected_effects[['model_name', 'level', 'outcome', 'term', 'coefficient', 'std_error', 'p_value', 'significance']], OUTPUT_TABLE_DIR / 'statistical_results_apa_table.csv')

    # 到这里为止，统计模型已经完成拟合并写出四类CSV结果。
    # coef_df保留完整系数，适合附录或进一步检查。
    # selected_effects只保留研究问题最关心的情绪和极化变量，适合正文表格和最后统一可视化。
    # fit_summary记录样本量、R方、伪R方、AIC和BIC，便于说明模型有效样本量。
    # statistical_results_apa_table.csv是报告正文可直接引用的简表。
    display(selected_effects.head(20))

except Exception as exc:
    warnings.warn(f'统计建模未完成：{exc}')
    raise


## 10.机器学习训练与解释

本节把传播热度问题转化为二分类预测任务：评论或帖子是否进入热度前20%。模型比较Baseline、LogisticRegression、RandomForest和HistGradientBoosting，并用averageprecision、ROC-AUC、F1等指标评估。特征重要性用于回答哪些变量更有助于预测高热度，不用于证明因果作用。

In [ ]:
# 本代码块目标：训练高热度预测模型，并解释不同特征组的预测贡献。
# 输入：评论层建模数据、帖子层建模数据和目标变量high_heat_top20。
# 输出：分类性能表、特征重要性表、特征组重要性表和重要性图表。
# 处理思路：统一预处理数值特征和分类特征，再比较多种分类模型。

# -----------------------------------------------------------------------------
# 10. 机器学习训练与解释
# -----------------------------------------------------------------------------


def train_ml_models(comment_df: pd.DataFrame, post_df: pd.DataFrame):
    # 机器学习部分承担预测任务：它关心哪些特征能更好地区分高热度和非高热度内容。
    # 这里把机器学习结果作为统计模型的补充：
    # 如果某类特征在预测中长期重要，说明它对平台互动热度有稳定的信息价值。
    # 评论层和帖子层分开训练，是因为两者的观测单位、变量含义和传播机制不同。
    """训练评论层和帖子层高热度分类模型。"""
    from sklearn.compose import ColumnTransformer
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    def make_preprocessor(numeric_features, categorical_features):
        # 数值变量和类别变量不能用同一种预处理方式：
        # 数值变量补中位数并标准化，避免量纲影响线性模型；
        # 类别变量补众数并独热编码，使subreddit、主题和主导情绪能进入同一个模型。
        numeric_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ])
        categorical_pipeline = Pipeline([
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ])
        return ColumnTransformer([
            ('num', numeric_pipeline, numeric_features),
            ('cat', categorical_pipeline, categorical_features),
        ])

    tasks = [
        {
            'task': 'M1',
            'level': 'comment',
            'data': comment_df,
            'target': 'high_heat_comment_top20',
            'numeric': ['negative_score', 'arousal_score', 'sentiment_valence', 'emotion_intensity', 'neutral_score', 'text_length_chars', 'word_count'],
            'categorical': ['subreddit', 'topic_id', 'dominant_emotion', 'has_url', 'has_question', 'has_exclamation'],
            'best_model': 'Logistic Regression',
        },
        {
            'task': 'M2',
            'level': 'post',
            'data': post_df,
            'target': 'high_heat_post_top20',
            'numeric': ['emotion_conflict_index', 'emotion_jsd_polarization', 'dominant_emotion_entropy', 'high_arousal_share', 'negative_share', 'mean_negative_score', 'mean_arousal_score', 'mean_sentiment_valence', 'share_replies_more_negative', 'share_replies_more_aroused'],
            'categorical': ['subreddit', 'dominant_topic_id'],
            'best_model': 'Random Forest',
        },
    ]

    performance_rows = []
    importances = {}
    group_rows = []

    for spec in tasks:
        df = spec['data'].copy()
        target = spec['target']
        numeric = [c for c in spec['numeric'] if c in df.columns]
        categorical = [c for c in spec['categorical'] if c in df.columns]
        features = numeric + categorical
        work = df[features + [target]].dropna(subset=[target]).copy()
        work[target] = work[target].astype(int)

        for col in categorical:
            work[col] = work[col].where(work[col].notna(), 'missing').astype(str)
            
        X = work[features]
        y = work[target]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )

        preprocessor = make_preprocessor(numeric, categorical)
        models = {
            'Baseline': DummyClassifier(strategy='most_frequent'),
            'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
            'Random Forest': RandomForestClassifier(n_estimators=250, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
            'HistGradientBoosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        }

        fitted = {}
        for model_name, estimator in models.items():
            pipeline = Pipeline([('preprocess', preprocessor), ('model', estimator)])
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            if hasattr(pipeline, 'predict_proba'):
                score = pipeline.predict_proba(X_test)[:, 1]
            else:
                score = pred
            performance_rows.append({
                'task': spec['task'],
                'level': spec['level'],
                'model_name': model_name,
                'n_train': len(X_train),
                'n_test': len(X_test),
                'positive_rate_train': y_train.mean(),
                'positive_rate_test': y_test.mean(),
                'accuracy': accuracy_score(y_test, pred),
                'precision': precision_score(y_test, pred, zero_division=0),
                'recall': recall_score(y_test, pred, zero_division=0),
                'f1': f1_score(y_test, pred, zero_division=0),
                'roc_auc': roc_auc_score(y_test, score),
                'average_precision': average_precision_score(y_test, score),
            })
            fitted[model_name] = pipeline

        best_pipeline = fitted[spec['best_model']]
        perm = permutation_importance(best_pipeline, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE, scoring='average_precision', n_jobs=-1)
        imp_df = pd.DataFrame({
            'task': spec['task'],
            'level': spec['level'],
            'model_name': spec['best_model'],
            'feature': features,
            'importance_mean': perm.importances_mean,
            'importance_std': perm.importances_std,
        }).sort_values('importance_mean', ascending=False)
        importances[spec['task']] = imp_df

        def group_for_feature(feature: str) -> str:
            if feature == 'subreddit':
                return 'subreddit'
            if 'topic' in feature:
                return 'topic'
            if feature in ['text_length_chars', 'word_count', 'has_url', 'has_question', 'has_exclamation']:
                return 'text'
            if feature in ['dominant_emotion']:
                return 'dominant_emotion'
            if feature in ['emotion_conflict_index', 'emotion_jsd_polarization', 'dominant_emotion_entropy', 'high_arousal_share', 'negative_share']:
                return 'polarization'
            if feature.startswith('mean_'):
                return 'mean_emotion'
            if feature.startswith('share_replies'):
                return 'reply_shift'
            return 'emotion'

        grouped = imp_df.assign(feature_group=imp_df['feature'].map(group_for_feature)).groupby('feature_group', as_index=False)['importance_mean'].sum()
        denom = grouped['importance_mean'].abs().sum()
        grouped['importance_share'] = grouped['importance_mean'] / denom if denom else 0
        for _, row in grouped.iterrows():
            group_rows.append({
                'task': spec['task'],
                'model_name': spec['best_model'],
                'feature_group': row['feature_group'],
                'total_importance': row['importance_mean'],
                'importance_share': row['importance_share'],
            })

    return pd.DataFrame(performance_rows), importances, pd.DataFrame(group_rows)


try:
    ml_performance, ml_importances, ml_group_importance = train_ml_models(comment_model, post_model)
    save_csv(ml_performance, OUTPUT_TABLE_DIR / 'ml_classification_performance.csv')
    save_csv(ml_importances['M1'], OUTPUT_TABLE_DIR / 'ml_comment_permutation_importance.csv')
    save_csv(ml_importances['M2'], OUTPUT_TABLE_DIR / 'ml_post_permutation_importance.csv')
    save_csv(ml_group_importance, OUTPUT_TABLE_DIR / 'ml_feature_group_importance.csv')

    # 机器学习结果先保存为CSV，再在最后统一生成图表。
    # 这样可以保证训练和评估已经完成，图表只是对已完成结果的表达。
    # ml_performance用于比较不同模型，ml_group_importance用于解释不同特征组贡献。
    # 两个permutation_importance表用于第12节绘制最终重要性图表。
    display(ml_performance)

except Exception as exc:
    warnings.warn(f'机器学习训练未完成：{exc}')
    raise


## 11.稳健性检验

本节检查结果是否过度依赖单一分析选择。检验包括替换高热度阈值、移除subreddit特征、替换热度指标和更换随机种子。若不同设置下结论方向仍相近，说明结果更稳定；若变化明显，则需要在报告讨论中说明限制。

In [ ]:
# 本代码块目标：检验主要预测结果是否依赖单一阈值、单一特征组或单一随机种子。
# 输入：评论层建模数据、帖子层建模数据和机器学习评估函数。
# 输出：阈值稳健性、移除subreddit、替换热度指标和随机种子结果表。
# 处理思路：每次只改变一个分析选择，观察averageprecision等指标是否大幅变化。

# -----------------------------------------------------------------------------
# 11. 稳健性检验
# -----------------------------------------------------------------------------


def robustness_checks(comment_df: pd.DataFrame, post_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # 稳健性检验用于检查结论是否过度依赖某个分析选择。
    # 本节检查四类选择：
    # 1)高热度阈值是否只能用top20%；
    # 2)去掉subreddit后模型是否明显变差；
    # 3)换一种热度指标后结论是否完全消失；
    # 4)换随机种子后模型表现是否剧烈波动。
    """运行主要稳健性检验。"""
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    def evaluate(df, target, numeric, categorical, model_name, seed=RANDOM_STATE):
        # 所有稳健性模型共用同一个评估函数。
        # 这样做有两个好处：
        # 1)不同检验之间只改变目标变量、特征组合或随机种子；
        # 2)预处理、训练、预测和指标计算保持一致，减少人为差异。
        features = [c for c in numeric + categorical if c in df.columns]
        work = df[features + [target]].dropna(subset=[target]).copy()
        work[target] = work[target].astype(int)

        active_numeric = [c for c in numeric if c in features]
        active_categorical = [c for c in categorical if c in features]

        for col in active_categorical:
            work[col] = work[col].where(work[col].notna(), 'missing').astype(str)

        X = work[features]
        y = work[target]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=.2, random_state=seed, stratify=y
        )

        pre = ColumnTransformer([
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), active_numeric),

            ('cat', Pipeline([
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), active_categorical),
        ])

        if model_name == 'Random Forest':
            model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced', random_state=seed, n_jobs=-1)
        else:
            model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
        pipe = Pipeline([('preprocess', pre), ('model', model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        score = pipe.predict_proba(X_test)[:, 1]
        return {
            'model_name': model_name,
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred, zero_division=0),
            'recall': recall_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, score),
            'average_precision': average_precision_score(y_test, score),
            'positive_rate_test': y_test.mean(),
        }

    comment_num = ['negative_score', 'arousal_score', 'sentiment_valence', 'emotion_intensity', 'neutral_score', 'text_length_chars', 'word_count']
    comment_cat = ['subreddit', 'topic_id', 'dominant_emotion', 'has_url', 'has_question', 'has_exclamation']
    post_num = ['emotion_conflict_index', 'emotion_jsd_polarization', 'dominant_emotion_entropy', 'high_arousal_share', 'negative_share', 'mean_negative_score', 'mean_arousal_score', 'mean_sentiment_valence', 'share_replies_more_negative', 'share_replies_more_aroused']
    post_cat = ['subreddit', 'dominant_topic_id']

    threshold_rows = []
    for q in [.90, .80, .75, .70]:
        suffix = f'top{int((1-q)*100)}'
        c = comment_df.copy()
        p = post_df.copy()
        c_target = f'high_heat_comment_{suffix}'
        p_target = f'high_heat_post_{suffix}'
        c[c_target] = c['comment_heat_composite'] >= c['comment_heat_composite'].quantile(q)
        p[p_target] = p['post_heat_log'] >= p['post_heat_log'].quantile(q)
        row = evaluate(c, c_target, comment_num, comment_cat, 'Logistic Regression')
        threshold_rows.append({'level': 'comment', 'threshold': suffix, 'target': c_target, **row})
        row = evaluate(p, p_target, post_num, post_cat, 'Random Forest')
        threshold_rows.append({'level': 'post', 'threshold': suffix, 'target': p_target, **row})

    without_rows = []
    row = evaluate(comment_df, 'high_heat_comment_top20', comment_num, comment_cat, 'Logistic Regression')
    with_comment_ap, with_comment_auc = row['average_precision'], row['roc_auc']
    without_rows.append({'level': 'comment', 'feature_setting': 'with_subreddit', **row, 'performance_drop_ap': 0, 'performance_drop_auc': 0})
    row = evaluate(post_df, 'high_heat_post_top20', post_num, post_cat, 'Random Forest')
    with_post_ap, with_post_auc = row['average_precision'], row['roc_auc']
    without_rows.append({'level': 'post', 'feature_setting': 'with_subreddit', **row, 'performance_drop_ap': 0, 'performance_drop_auc': 0})
    row = evaluate(comment_df, 'high_heat_comment_top20', comment_num, [c for c in comment_cat if c != 'subreddit'], 'Logistic Regression')
    without_rows.append({'level': 'comment', 'feature_setting': 'without_subreddit', **row, 'performance_drop_ap': with_comment_ap - row['average_precision'], 'performance_drop_auc': with_comment_auc - row['roc_auc']})
    row = evaluate(post_df, 'high_heat_post_top20', post_num, [c for c in post_cat if c != 'subreddit'], 'Random Forest')
    without_rows.append({'level': 'post', 'feature_setting': 'without_subreddit', **row, 'performance_drop_ap': with_post_ap - row['average_precision'], 'performance_drop_auc': with_post_auc - row['roc_auc']})

    alt_rows = []
    for outcome in ['comment_heat_composite', 'comment_upvotes_num', 'reply_count_num']:
        c = comment_df.copy()
        target = f'high_{outcome}_top20'
        c[target] = c[outcome] >= c[outcome].quantile(.80)
        alt_rows.append({'level': 'comment', 'heat_metric': outcome, 'target': target, **evaluate(c, target, comment_num, comment_cat, 'Logistic Regression')})
    for outcome in ['post_heat_log', 'engagement_heat', 'comment_discussion_heat']:
        p = post_df.copy()
        target = f'high_{outcome}_top20'
        p[target] = p[outcome] >= p[outcome].quantile(.80)
        alt_rows.append({'level': 'post', 'heat_metric': outcome, 'target': target, **evaluate(p, target, post_num, post_cat, 'Random Forest')})

    seed_rows = []
    for seed in [1, 7, 21, 42, 99]:
        seed_rows.append({'level': 'comment', 'seed': seed, **evaluate(comment_df, 'high_heat_comment_top20', comment_num, comment_cat, 'Logistic Regression', seed=seed)})
        seed_rows.append({'level': 'post', 'seed': seed, **evaluate(post_df, 'high_heat_post_top20', post_num, post_cat, 'Random Forest', seed=seed)})

    return pd.DataFrame(threshold_rows), pd.DataFrame(without_rows), pd.DataFrame(alt_rows), pd.DataFrame(seed_rows)


try:
    threshold_perf, without_subreddit, alt_heat, seed_perf = robustness_checks(comment_model, post_model)
    save_csv(threshold_perf, OUTPUT_TABLE_DIR / 'robustness_threshold_performance.csv')
    save_csv(without_subreddit, OUTPUT_TABLE_DIR / 'robustness_without_subreddit.csv')
    save_csv(alt_heat, OUTPUT_TABLE_DIR / 'robustness_alternative_heat_metrics.csv')
    save_csv(seed_perf, OUTPUT_TABLE_DIR / 'robustness_random_seed_summary.csv')

    conclusion_rows = [
        {'conclusion': '评论层高唤醒情绪对高热度预测有稳定贡献', 'evidence': '阈值和随机种子检验下评论层模型保持可用预测力。', 'robustness': 'moderately robust'},
        {'conclusion': '帖子层情感冲突指数对高热度预测有贡献', 'evidence': '帖子层极化特征组在机器学习和阈值检验中有预测贡献。', 'robustness': 'moderately robust'},
        {'conclusion': 'subreddit是最强结构性预测因素', 'evidence': '去除subreddit后comment/post AP均下降，帖子层下降更明显。', 'robustness': 'robust'},
        {'conclusion': '结果不只依赖top20 阈值', 'evidence': 'top10、top20、top25、top30 下模型仍有预测能力。', 'robustness': 'not solely dependent on top20'},
    ]
    save_csv(pd.DataFrame(conclusion_rows), OUTPUT_TABLE_DIR / 'robustness_conclusion_summary.csv')

    # 稳健性检验到这里已经完成四类结果表保存。
    # threshold_perf用于观察高热度阈值变化是否影响模型表现。
    # without_subreddit用于观察社区特征被移除后预测能力是否下降。
    # alt_heat用于检查替换热度指标后的结果是否保持方向一致。
    # seed_perf用于检查随机种子变化下模型表现是否稳定。
    display(without_subreddit)

except Exception as exc:
    warnings.warn(f'稳健性检验未完成：{exc}')
    raise

## 12.结果可视化与图表导出

本节放在描述统计、统计建模、机器学习训练和稳健性检验之后，专门负责把已经完成的分析结果转成图表。这样图表的逻辑来源更清楚：描述性图表来自第8节统计表和建模数据，系数图来自第9节统计模型结果，特征重要性图来自第10节机器学习结果，稳健性图来自第11节检验结果。

图表类型包括条形图、热力图、雷达图、堆叠条形图、气泡图、棒棒糖图、箱线图、小提琴图、散点趋势图、系数图、重要性条形图、折线图和斜率图。

In [ ]:
# 本代码块目标：在所有分析完成之后统一生成正式图表。
# 输入：第8节描述统计数据、第9节统计模型结果、第10节机器学习结果、第11节稳健性结果。
# 输出：outputs/figures/中的正式PNG图表和outputs/tables/figure_index.csv图表索引。
# 处理思路：每张图都只把已有分析结果可视化，不在绘图阶段重新训练模型或改变结论。
# -----------------------------------------------------------------------------
# 12.结果可视化与图表导出
# -----------------------------------------------------------------------------

import csv
import matplotlib.pyplot as plt

# 统一matplotlib输出参数。
# figure.dpi控制notebook内显示清晰度，savefig.dpi控制导出PNG清晰度。
# axes.unicode_minus=False避免负号在部分中文字体环境中显示异常。
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 220
plt.rcParams['axes.unicode_minus'] = False


def save_figure(path: Path) -> None:
    # 所有图表都通过这个函数保存，确保输出目录存在、边距紧凑、画布及时关闭。
    # 统一保存逻辑可以减少重复代码，也能避免多张图连续生成时互相污染。
    """统一保存图表并关闭画布。"""
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f'Saved figure: {path}')


def wrap_labels(labels, width=38):
    # 报告中的主题名很长，如果完整放进坐标轴会挤压图形主体。
    # 这里做保守截断，只影响图表标签显示，不影响底层数据和结果表。
    return [str(x)[:width] for x in labels]


def plot_descriptive_figures(comment_df: pd.DataFrame, post_df: pd.DataFrame) -> None:
    # 本函数生成描述性图表。
    # 这些图不依赖模型系数或机器学习结果，只展示样本结构、情绪结构和变量分布。
    # 因为它们仍然属于“描述统计”，所以逻辑上可以解释为第8节统计表的视觉版本。
    subreddit_counts = comment_df['subreddit'].value_counts().sort_values()
    plt.figure(figsize=(8, 4.6))
    plt.barh(subreddit_counts.index, subreddit_counts.values, color='#3B6EA8')
    plt.xlabel('Comment count')
    plt.title('Comment Count by Subreddit')
    save_figure(OUTPUT_FIGURE_DIR / 'fig01_subreddit_comment_count.png')

    # 主题数量图用横向条形图，因为主题标签比数值更长。
    # 横向布局可以让读者先读主题，再看评论数量。
    topic_counts = comment_df.groupby(['topic_id', 'topic_label_en']).size().sort_values()
    labels = [f'{idx[0]}:{idx[1][:36]}' for idx in topic_counts.index]
    plt.figure(figsize=(9, 5.2))
    plt.barh(labels, topic_counts.values, color='#4C956C')
    plt.xlabel('Comment count')
    plt.title('Comment Count by Topic')
    save_figure(OUTPUT_FIGURE_DIR / 'fig02_topic_comment_count.png')

    # 情绪热力图适合表达“主题×情绪变量”的矩阵。
    # 行是主题，列是情绪得分，颜色用于帮助读者快速定位高低差异。
    heatmap_cols = ['positive_score', 'negative_score', 'arousal_score', 'sentiment_valence']
    emotion_by_topic = comment_df.groupby('topic_label_en')[heatmap_cols].mean().sort_index()
    plt.figure(figsize=(9.5, 5.5))
    plt.imshow(emotion_by_topic.values, aspect='auto', cmap='RdYlBu_r')
    plt.colorbar(label='Mean score')
    plt.yticks(range(len(emotion_by_topic)), wrap_labels(emotion_by_topic.index, 42), fontsize=7)
    plt.xticks(range(len(heatmap_cols)), heatmap_cols, rotation=30, ha='right')
    plt.title('Average Emotion Scores by Topic')
    save_figure(OUTPUT_FIGURE_DIR / 'fig03_topic_emotion_heatmap.png')

    # 雷达图只选评论量最高的5个主题。
    # 如果把10个主题全部放入雷达图，线条会互相覆盖，反而降低可读性。
    # 这里的数值做0-1缩放，用于比较轮廓，原始概率大小在描述统计表中呈现。
    radar_cols = ['positive_score', 'negative_score', 'arousal_score', 'emotion_intensity']
    radar_source = (
        comment_df.groupby('topic_label_en')[radar_cols + ['post_id']]
        .agg({**{c: 'mean' for c in radar_cols}, 'post_id': 'size'})
        .rename(columns={'post_id': 'comment_count'})
        .sort_values('comment_count', ascending=False)
        .head(5)
    )
    radar_values = radar_source[radar_cols].copy()
    radar_min = radar_values.min()
    radar_range = (radar_values.max() - radar_min).replace(0, 1)
    radar_scaled = (radar_values - radar_min) / radar_range
    angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
    angles += angles[:1]
    fig = plt.figure(figsize=(7.2, 6.2))
    ax = plt.subplot(111, polar=True)
    for topic_name, row in radar_scaled.iterrows():
        values = row.tolist() + row.tolist()[:1]
        ax.plot(angles, values, linewidth=1.6, label=topic_name[:26])
        ax.fill(angles, values, alpha=.08)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_cols, fontsize=8)
    ax.set_yticklabels([])
    ax.set_title('Emotion Profile Radar for High-Volume Topics', pad=18)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.08), fontsize=7)
    save_figure(OUTPUT_FIGURE_DIR / 'fig04_topic_emotion_profile_radar.png')

    # 堆叠条形图比较不同subreddit的情绪组成。
    # 为了让不同社区可比，先把四个情绪得分除以行总和，转为相对组成。
    stack_source = (
        comment_df.groupby('subreddit')[['positive_score', 'negative_score', 'neutral_score', 'arousal_score']]
        .mean()
        .loc[comment_df['subreddit'].value_counts().head(6).index]
    )
    stack = stack_source.div(stack_source.sum(axis=1), axis=0).sort_values('negative_score')
    plt.figure(figsize=(8.6, 4.8))
    left = np.zeros(len(stack))
    colors = ['#5AA469', '#C44E52', '#9E9E9E', '#F2A65A']
    for col, color in zip(stack.columns, colors):
        plt.barh(stack.index, stack[col], left=left, label=col, color=color)
        left += stack[col].values
    plt.xlabel('Share within selected emotion scores')
    plt.title('Emotion Composition by Subreddit')
    plt.legend(loc='lower right', fontsize=7)
    save_figure(OUTPUT_FIGURE_DIR / 'fig05_subreddit_emotion_composition_stacked.png')

    # 气泡图同时表达四个信息：横轴是情感冲突，纵轴是热度，气泡大小是帖子数，颜色是负向占比。
    # 这类图适合做模式观察，正式关系判断仍依赖第9节统计模型。
    bubble_topic = (
        post_df.groupby('dominant_topic_label_en')
        .agg(
            post_count=('post_id', 'size'),
            mean_post_heat_log=('post_heat_log', 'mean'),
            mean_emotion_conflict_index=('emotion_conflict_index', 'mean'),
            mean_negative_share=('negative_share', 'mean'),
        )
        .dropna()
    )
    bubble_size = 120 + 900 * bubble_topic['post_count'] / bubble_topic['post_count'].max()
    plt.figure(figsize=(8.2, 5.2))
    points = plt.scatter(
        bubble_topic['mean_emotion_conflict_index'],
        bubble_topic['mean_post_heat_log'],
        s=bubble_size,
        c=bubble_topic['mean_negative_share'],
        cmap='viridis',
        alpha=.72,
        edgecolor='white',
        linewidth=.8,
    )
    for topic_name, row in bubble_topic.iterrows():
        plt.text(row['mean_emotion_conflict_index'], row['mean_post_heat_log'], topic_name[:18], fontsize=6, ha='center', va='center')
    plt.colorbar(points, label='Mean negative share')
    plt.xlabel('Mean emotion conflict index')
    plt.ylabel('Mean post heat(log)')
    plt.title('Topic Conflict, Heat, and Size')
    save_figure(OUTPUT_FIGURE_DIR / 'fig06_topic_heat_conflict_bubble.png')

    # 棒棒糖图比普通条形图更轻，适合主题均值排序。
    topic_heat = post_df.groupby('dominant_topic_label_en')['post_heat_log'].mean().sort_values()
    plt.figure(figsize=(9, 5.2))
    plt.hlines(range(len(topic_heat)), 0, topic_heat.values, color='#A7B6C2')
    plt.plot(topic_heat.values, range(len(topic_heat)), 'o', color='#D95F02')
    plt.yticks(range(len(topic_heat)), wrap_labels(topic_heat.index, 42), fontsize=7)
    plt.xlabel('Mean post heat(log)')
    plt.title('Average Post Heat by Topic')
    save_figure(OUTPUT_FIGURE_DIR / 'enhanced_topic_post_heat_lollipop.png')

    # 情感冲突排序图用于展示不同主题下评论情绪冲突程度差异。
    topic_conflict = post_df.groupby('dominant_topic_label_en')['emotion_conflict_index'].mean().sort_values()
    plt.figure(figsize=(9, 5.2))
    plt.hlines(range(len(topic_conflict)), 0, topic_conflict.values, color='#A7B6C2')
    plt.plot(topic_conflict.values, range(len(topic_conflict)), 'o', color='#7570B3')
    plt.yticks(range(len(topic_conflict)), wrap_labels(topic_conflict.index, 42), fontsize=7)
    plt.xlabel('Mean emotion conflict index')
    plt.title('Emotion Conflict Index by Topic')
    save_figure(OUTPUT_FIGURE_DIR / 'enhanced_topic_emotion_conflict_rank.png')

    # 箱线图展示不同主题下帖子热度的分布，比单独查看均值更完整。
    # showfliers=False是为了避免极端高热度帖子把主体箱体压扁。
    top_topics = post_df['dominant_topic_label_en'].value_counts().head(8).index
    box_data = [post_df.loc[post_df['dominant_topic_label_en'] == topic, 'post_heat_log'].dropna() for topic in top_topics]
    plt.figure(figsize=(9.4, 5.2))
    plt.boxplot(box_data, vert=False, patch_artist=True, showfliers=False, boxprops={'facecolor': '#D6E6F2', 'color': '#355C7D'}, medianprops={'color': '#C44E52'})
    plt.yticks(range(1, len(top_topics) + 1), wrap_labels(top_topics, 36), fontsize=7)
    plt.xlabel('Post heat(log)')
    plt.title('Post Heat Distribution by Dominant Topic')
    save_figure(OUTPUT_FIGURE_DIR / 'fig08_post_heat_topic_boxplot.png')

    # 散点图展示连续变量之间的整体趋势。
    # 样本较多时抽样2500个点，避免图像过密，同时固定随机种子保证每次抽样稳定。
    sample = post_df[['emotion_conflict_index', 'post_heat_log']].dropna()
    if len(sample) > 2500:
        sample = sample.sample(2500, random_state=RANDOM_STATE)
    plt.figure(figsize=(6.5, 4.8))
    plt.scatter(sample['emotion_conflict_index'], sample['post_heat_log'], s=10, alpha=.35, color='#2C7FB8')
    z = np.polyfit(sample['emotion_conflict_index'], sample['post_heat_log'], 1)
    x = np.linspace(sample['emotion_conflict_index'].min(), sample['emotion_conflict_index'].max(), 100)
    plt.plot(x, z[0] * x + z[1], color='#D95F02', linewidth=2)
    plt.xlabel('Emotion conflict index')
    plt.ylabel('Post heat(log)')
    plt.title('Emotion Conflict Index and Post Heat')
    save_figure(OUTPUT_FIGURE_DIR / 'fig07_emotion_conflict_vs_post_heat.png')

    # 小提琴图展示评论热度在不同社区中的分布形状。
    # 每个社区最多抽样800条评论，避免大社区完全支配绘图速度和图形密度。
    violin_subs = comment_df['subreddit'].value_counts().head(6).index
    violin_data = []
    for sub in violin_subs:
        values = comment_df.loc[comment_df['subreddit'] == sub, 'comment_heat_composite'].dropna()
        n = min(800, values.shape[0])
        violin_data.append(values.sample(n, random_state=RANDOM_STATE))
    plt.figure(figsize=(8.8, 4.8))
    parts = plt.violinplot(violin_data, showmeans=False, showmedians=True, vert=False)
    for body in parts['bodies']:
        body.set_facecolor('#8DA0CB')
        body.set_edgecolor('#4A5568')
        body.set_alpha(.72)
    plt.yticks(range(1, len(violin_subs) + 1), violin_subs, fontsize=8)
    plt.xlabel('Comment heat composite')
    plt.title('Comment Heat Distribution by Subreddit')
    save_figure(OUTPUT_FIGURE_DIR / 'fig11_comment_heat_subreddit_violin.png')

    # 直方图用于说明热度变量偏态分布。
    plt.figure(figsize=(6.5, 4.2))
    plt.hist(comment_df['comment_heat_composite'].dropna(), bins=40, color='#5B8DB8', edgecolor='white')
    plt.xlabel('Comment heat composite')
    plt.ylabel('Count')
    plt.title('Distribution of Comment-Level Heat')
    save_figure(OUTPUT_FIGURE_DIR / 'fig09_comment_heat_distribution.png')

    plt.figure(figsize=(6.5, 4.2))
    plt.hist(post_df['post_heat_log'].dropna(), bins=40, color='#70A37F', edgecolor='white')
    plt.xlabel('Post heat(log)')
    plt.ylabel('Count')
    plt.title('Distribution of Post-Level Heat')
    save_figure(OUTPUT_FIGURE_DIR / 'fig10_post_heat_distribution.png')

    # 相关矩阵用于补充说明变量之间的线性相关结构。
    corr_cols = ['post_heat_log', 'engagement_heat', 'emotion_conflict_index', 'dominant_emotion_entropy', 'high_arousal_share', 'negative_share']
    corr = post_df[[c for c in corr_cols if c in post_df.columns]].corr()
    plt.figure(figsize=(7, 5.8))
    plt.imshow(corr.values, vmin=-1, vmax=1, cmap='RdBu_r')
    plt.colorbar(label='Pearson r')
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right', fontsize=7)
    plt.yticks(range(len(corr.index)), corr.index, fontsize=7)
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
    plt.title('Correlation Matrix of Emotion, Polarization, and Heat Metrics')
    save_figure(OUTPUT_FIGURE_DIR / 'fig12_post_correlation_heatmap.png')


def plot_statistical_figures(selected_df: pd.DataFrame) -> None:
    # 系数图只使用第9节已经拟合好的selected_effects。
    # 这里不重新拟合模型，只把系数和95%置信区间画出来。
    def plot_selected_coefficients(level: str, path: Path, title: str) -> None:
        plot_df = selected_df[selected_df['level'] == level].copy()
        plot_df = plot_df[plot_df['model_name'].isin(['C1', 'P1', 'P2'])]
        if plot_df.empty:
            return
        labels = plot_df['model_name'] + ':' + plot_df['term']
        y = np.arange(len(plot_df))
        plt.figure(figsize=(8.5, max(4, len(plot_df) * .35)))
        plt.axvline(0, color='black', linewidth=.8)
        xerr = [plot_df['coefficient'] - plot_df['ci_low'], plot_df['ci_high'] - plot_df['coefficient']]
        plt.errorbar(plot_df['coefficient'], y, xerr=xerr, fmt='o', color='#2C7FB8', ecolor='#8DA0CB', capsize=3)
        plt.yticks(y, labels, fontsize=7)
        plt.xlabel('Coefficient with 95% CI')
        plt.title(title)
        save_figure(path)

    plot_selected_coefficients('comment', OUTPUT_FIGURE_DIR / 'fig13_comment_level_selected_coefficients.png', 'Comment-Level Emotion Predictors')
    plot_selected_coefficients('post', OUTPUT_FIGURE_DIR / 'fig14_post_level_selected_coefficients.png', 'Post-Level Polarization Predictors')


def plot_machine_learning_figures(importance_map: dict[str, pd.DataFrame]) -> None:
    # 特征重要性图只使用第10节permutationimportance结果。
    # 重要性越高，说明打乱该特征后averageprecision下降越多。
    def plot_importance(imp_df: pd.DataFrame, path: Path, title: str) -> None:
        top = imp_df.head(14).sort_values('importance_mean')
        plt.figure(figsize=(7.5, 4.8))
        plt.barh(top['feature'], top['importance_mean'], color='#3B6EA8')
        plt.xlabel('Permutation importance(averageprecision)')
        plt.title(title)
        save_figure(path)

    plot_importance(importance_map['M1'], OUTPUT_FIGURE_DIR / 'fig18_ml_comment_permutation_importance.png', 'Top Predictors of High-Heat Comments')
    plot_importance(importance_map['M2'], OUTPUT_FIGURE_DIR / 'fig19_ml_post_permutation_importance.png', 'Top Predictors of High-Heat Posts')


def plot_robustness_figures(threshold_df: pd.DataFrame, without_subreddit_df: pd.DataFrame) -> None:
    # 稳健性图只使用第11节检验结果。
    # 第一张折线图比较不同高热度阈值下模型表现是否稳定。
    plt.figure(figsize=(7, 4.6))
    for level, group in threshold_df.groupby('level'):
        plt.plot(group['threshold'], group['average_precision'], marker='o', label=level)
    plt.ylabel('Average precision')
    plt.xlabel('High-heat threshold')
    plt.title('Robustness Across High-Heat Thresholds')
    plt.legend()
    save_figure(OUTPUT_FIGURE_DIR / 'fig23_robustness_threshold_performance.png')

    # 第二张斜率图比较保留和移除subreddit特征后的预测表现。
    # without_subreddit_df是长表，所以先pivot成一行一个level、一列一个feature_setting。
    # 这样绘图代码直接读取with_subreddit和without_subreddit两列，不依赖行顺序。
    # 如果移除后明显下降，说明社区结构对高热度预测贡献很大。
    pivot = without_subreddit_df.pivot(index='level', columns='feature_setting', values='average_precision')
    required_cols = ['with_subreddit', 'without_subreddit']
    missing_cols = [c for c in required_cols if c not in pivot.columns]
    if missing_cols:
        raise KeyError('稳健性表缺少绘图所需列：' + ', '.join(missing_cols))
    plt.figure(figsize=(6.5, 4.4))
    for level, row in pivot.iterrows():
        plt.plot(required_cols, [row['with_subreddit'], row['without_subreddit']], marker='o', label=level)
    plt.ylabel('Average precision')
    plt.title('Effect of Removing Subreddit Features')
    plt.legend()
    save_figure(OUTPUT_FIGURE_DIR / 'fig24_robustness_without_subreddit.png')


def write_figure_index() -> None:
    # 图表索引让报告写作和文件检查更方便。
    # 这里保存的是图表元数据，不保存任何额外解释性Markdown。
    rows = [
        ('fig01_subreddit_comment_count', 'outputs/figures/fig01_subreddit_comment_count.png', 'Comment Count by Subreddit', 'horizontal bar chart'),
        ('fig02_topic_comment_count', 'outputs/figures/fig02_topic_comment_count.png', 'Comment Count by Topic', 'horizontal bar chart'),
        ('fig03_topic_emotion_heatmap', 'outputs/figures/fig03_topic_emotion_heatmap.png', 'Average Emotion Scores by Topic', 'annotated heatmap'),
        ('fig04_topic_emotion_profile_radar', 'outputs/figures/fig04_topic_emotion_profile_radar.png', 'Emotion Profile Radar for High-Volume Topics', 'radar chart'),
        ('fig05_subreddit_emotion_composition_stacked', 'outputs/figures/fig05_subreddit_emotion_composition_stacked.png', 'Emotion Composition by Subreddit', 'stacked horizontal bar chart'),
        ('fig06_topic_heat_conflict_bubble', 'outputs/figures/fig06_topic_heat_conflict_bubble.png', 'Topic Conflict, Heat, and Size', 'bubble chart'),
        ('enhanced_topic_post_heat_lollipop', 'outputs/figures/enhanced_topic_post_heat_lollipop.png', 'Average Post Heat by Topic', 'lollipop plot'),
        ('enhanced_topic_emotion_conflict_rank', 'outputs/figures/enhanced_topic_emotion_conflict_rank.png', 'Emotion Conflict Index by Topic', 'ranked lollipop plot'),
        ('fig07_emotion_conflict_vs_post_heat', 'outputs/figures/fig07_emotion_conflict_vs_post_heat.png', 'Emotion Conflict Index and Post Heat', 'scatter plot with trend line'),
        ('fig08_post_heat_topic_boxplot', 'outputs/figures/fig08_post_heat_topic_boxplot.png', 'Post Heat Distribution by Dominant Topic', 'box plot'),
        ('fig09_comment_heat_distribution', 'outputs/figures/fig09_comment_heat_distribution.png', 'Distribution of Comment-Level Heat', 'histogram'),
        ('fig10_post_heat_distribution', 'outputs/figures/fig10_post_heat_distribution.png', 'Distribution of Post-Level Heat', 'histogram'),
        ('fig11_comment_heat_subreddit_violin', 'outputs/figures/fig11_comment_heat_subreddit_violin.png', 'Comment Heat Distribution by Subreddit', 'violin plot'),
        ('fig12_post_correlation_heatmap', 'outputs/figures/fig12_post_correlation_heatmap.png', 'Correlation Matrix of Emotion, Polarization, and Heat Metrics', 'annotated correlation heatmap'),
        ('fig13_comment_level_selected_coefficients', 'outputs/figures/fig13_comment_level_selected_coefficients.png', 'Comment-Level Emotion Predictors', 'coefficient plot'),
        ('fig14_post_level_selected_coefficients', 'outputs/figures/fig14_post_level_selected_coefficients.png', 'Post-Level Polarization Predictors', 'coefficient plot'),
        ('fig18_ml_comment_permutation_importance', 'outputs/figures/fig18_ml_comment_permutation_importance.png', 'Top Predictors of High-Heat Comments', 'horizontal importance bar chart'),
        ('fig19_ml_post_permutation_importance', 'outputs/figures/fig19_ml_post_permutation_importance.png', 'Top Predictors of High-Heat Posts', 'horizontal importance bar chart'),
        ('fig23_robustness_threshold_performance', 'outputs/figures/fig23_robustness_threshold_performance.png', 'Robustness Across High-Heat Thresholds', 'line chart'),
        ('fig24_robustness_without_subreddit', 'outputs/figures/fig24_robustness_without_subreddit.png', 'Effect of Removing Subreddit Features', 'slope chart'),
    ]
    with (OUTPUT_TABLE_DIR / 'figure_index.csv').open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['figure_id', 'file_path', 'title', 'chart_type'])
        writer.writerows(rows)


# 按逻辑顺序执行绘图：先描述性图表，再模型系数图，再机器学习图，再稳健性图。
# 每一步都只读取已经完成的分析结果，不在绘图阶段修改模型或重新定义指标。
plot_descriptive_figures(comment_model, post_model)
plot_statistical_figures(selected_effects)
plot_machine_learning_figures(ml_importances)
plot_robustness_figures(threshold_perf, without_subreddit)
write_figure_index()

## 13.最终完整性检查

本节检查主数据、GoEmotions标签、本地模型、中间结果、建模数据、关键表格、关键图表和最终报告是否齐全。如果检查通过，说明读者可以沿着前面各节理解并运行整套流程。

In [ ]:
# 本代码块目标：检查关键输入、中间结果、建模数据、主要CSV和主要图表是否齐全。
# 输入：本notebook前面生成或读取的各类文件路径。
# 输出：文件存在性表、文件大小表和最终样本量摘要。
# 处理思路：只检查Notebook自身运行链路需要的文件，不检查已经精简掉的脚本或额外Markdown。

# -----------------------------------------------------------------------------
# 13.最终完整性检查
# -----------------------------------------------------------------------------
# required_files分成五类：
# 1)主数据和GoEmotions标签文件；
# 2)本地情绪模型权重；
# 3)主流程中间parquet和最终建模数据；
# 4)统计建模、机器学习和稳健性CSV结果；
# 5)第12节统一导出的关键图表。

required_files = [
    MAIN_DATA_PATH,
    GOEMOTIONS_DIR / 'train.tsv',
    GOEMOTIONS_DIR / 'dev.tsv',
    GOEMOTIONS_DIR / 'test.tsv',
    GOEMOTIONS_DIR / 'emotions.txt',
    GOEMOTIONS_DIR / 'sentiment_mapping.json',
    GOEMOTIONS_DIR / 'ekman_mapping.json',
    MODEL_DIR / 'model.safetensors',
    COMMENTS_FLAT_PATH,
    REPLIES_FLAT_PATH,
    COMMENTS_CLEAN_PATH,
    REPLIES_CLEAN_PATH,
    COMMENTS_EMOTION_PATH,
    REPLIES_EMOTION_PATH,
    COMMENTS_TOPIC_PATH,
    POST_POLARIZATION_PATH,
    COMMENT_MODEL_PATH,
    POST_MODEL_PATH,
    OUTPUT_TABLE_DIR / 'descriptive_subreddit_comment_summary.csv',
    OUTPUT_TABLE_DIR / 'descriptive_topic_comment_summary.csv',
    OUTPUT_TABLE_DIR / 'statistical_selected_effects_summary.csv',
    OUTPUT_TABLE_DIR / 'ml_classification_performance.csv',
    OUTPUT_TABLE_DIR / 'robustness_threshold_performance.csv',
    OUTPUT_TABLE_DIR / 'figure_index.csv',
    OUTPUT_FIGURE_DIR / 'fig01_subreddit_comment_count.png',
    OUTPUT_FIGURE_DIR / 'fig04_topic_emotion_profile_radar.png',
    OUTPUT_FIGURE_DIR / 'fig05_subreddit_emotion_composition_stacked.png',
    OUTPUT_FIGURE_DIR / 'fig06_topic_heat_conflict_bubble.png',
    OUTPUT_FIGURE_DIR / 'fig08_post_heat_topic_boxplot.png',
    OUTPUT_FIGURE_DIR / 'fig13_comment_level_selected_coefficients.png',
    OUTPUT_FIGURE_DIR / 'fig23_robustness_threshold_performance.png',
]

# 文件检查表把路径、存在状态和文件大小放在一起。
# 如果某个关键文件缺失，读者可以直接看到是哪一类产物没有生成。
check = pd.DataFrame({
    'file_path': [str(path) for path in required_files],
    'exists': [Path(path).exists() for path in required_files],
    'file_size_mb': [file_size_mb(path) for path in required_files],
})
display(check)

# 只要有关键文件缺失，就明确报错。
# 这样不会出现“Notebook跑完了但其实结果不完整”的假象。
missing = check.loc[~check['exists'], 'file_path'].tolist()
if missing:
    raise FileNotFoundError('仍有关键文件缺失：' + ', '.join(missing))

# 样本量摘要用于最后复核各阶段数据规模是否合理。
# 如果某次运行后行数突然大幅变化，通常说明上游清洗或展开逻辑发生了变化。
summary = {
    'comments_clean_rows': len(comments_clean),
    'replies_clean_rows': len(replies_clean),
    'comments_with_emotions_rows': len(comments_emotions),
    'comments_with_topics_rows': len(comments_topics),
    'post_level_rows': len(post_polar),
    'comment_model_rows': len(comment_model),
    'post_model_rows': len(post_model),
    'unique_subreddits': int(comment_model['subreddit'].nunique()) if 'subreddit' in comment_model.columns else None,
    'emotion_probability_columns': len(emotion_prob_cols),
    'topic_count_in_model': int(comment_model['topic_id'].nunique()) if 'topic_id' in comment_model.columns else None,
}
summary_df = pd.DataFrame([summary])
display(summary_df)

print('主流程notebook运行完成：数据处理、建模训练、稳健性检验、结果可视化和完整性检查已执行。')


## 完成

主流程已覆盖数据读取、评论展开、文本清洗、情绪识别、主题建模、极化指标、热度指标、描述统计、统计建模、机器学习训练、稳健性检验、结果可视化和完整性检查。逻辑顺序是先生成数据和结果，再统一导出图表。

最终结果位置：

-建模数据：`processed_data/`
-结果表格：`outputs/tables/`
-结果图表：`outputs/figures/`
-分析报告：`report/`


## 思路细节和具体的AI使用说明

本项目在完成过程中使用了AI工具作为学习、辅导和代码优化辅助工具。AI在我确定研究方向、理解数据内容、运行和检查代码的过程中，帮助我完善分析流程、改进代码结构、补充解释文字，并提醒我注意研究边界。

### 1. 项目整体结构设计

AI帮助我把原本比较分散的数据处理和分析步骤整理成一个完整的数据分析pipeline。最终Notebook按照数据读取、嵌套评论展开、文本清洗、情绪识别、主题建模、帖子层情感极化指标、传播热度建模、统计模型、机器学习模型、稳健性检验、结果可视化和完整性检查的顺序展开。

通过这个过程，我学习到数据项目需要先保证数据链路稳定，再进行建模和可视化，不能一开始就直接画图或训练模型。

### 2. 代码可复现性和工程化细节

AI帮助我改进了Notebook中很多和可复现性有关的代码细节，包括：

- 使用 `Path` 和相对路径管理项目文件，避免写死个人电脑上的绝对路径。
- 集中配置主数据、GoEmotions 目录、本地模型目录、中间数据目录、结果表格目录和图表目录。
- 使用 `processed_data`、`outputs/tables` 和 `outputs/figures` 统一管理输出文件。
- 在关键阶段保存 parquet 中间结果，避免每次重新运行都重复完成耗时步骤。
- 对情绪识别这种耗时操作设置“如果正式结果已经存在则优先读取”的逻辑。
- 使用统一的保存函数保存CSV和parquet文件，减少重复代码。
- 在最后增加完整性检查，确认输入文件、中间结果、建模数据、结果表格和图表是否齐全。


### 3. 嵌套评论数据展开

原始Reddit数据中的`comments`字段采用嵌套结构。这里整理了将帖子、评论和回复拆分成结构化表格的处理思路。

包括：

- 使用`normalize_nested_object`处理list、dict、numpy array、字符串化JSON等不同形态的嵌套对象。
- 使用`get_first_existing`从多个候选字段名中提取可用字段，以适应不同数据来源字段命名不完全一致的问题。
- 在展开评论和回复时保留`post_id`、`comment_id`等连接键，使后续可以把评论级、回复级和帖子级数据重新关联起来。
- 将顶层评论和回复分别保存成`comments_flat_raw.parquet`和`replies_flat_raw.parquet`，为后续清洗、情绪识别和聚合分析提供基础。

### 4. 文本清洗和特征构造

AI帮助我完善了文本清洗部分，使清洗逻辑更加稳健，同时尽量避免过度清洗导致语义损失。

相关细节包括：

- 使用`clean_text_basic`统一处理空值、首尾空白和连续空白。
- 使用`is_deleted_or_removed`识别Reddit中常见的 `[deleted]`、`[removed]` 等无效文本。
- 将互动数字字段安全转换为数值，避免字符串、缺失值或异常值影响后续模型。
- 构造文本长度、词数、是否包含 URL、是否包含问号、是否包含感叹号等表达形式特征。
- 在清洗时尽量只删除明显不可分析的文本。

### 5. GoEmotions情绪识别

AI 帮助我理解了如何把GoEmotions模型输出的多标签情绪概率转化为适合研究解释的变量。

包括：

- 使用 `BATCH_SIZE`、`MAX_LENGTH` 和 `CHUNK_SIZE` 控制推理速度和内存占用。
- 如果情绪识别结果已经存在，就直接读取保存好的parquet文件，避免重复进行耗时推理。
- 将GoEmotions的具体情绪标签整理为正向情绪、负向情绪、高唤醒情绪等更容易解释的组合变量。
- 构造情绪效价、情绪强度、主导情绪等变量，使情绪分析不只停留在单个标签层面。

AI在这里帮助我理解了模型输出和研究变量之间的关系。模型给出的只是概率，真正用于研究时还需要结合研究问题进行变量整理和解释。

### 6. 主题建模

TF-IDF和NMF用于识别评论讨论的主要内容结构，并在后续模型中作为控制变量使用。

### 7. 帖子层情感极化指标

这是本项目中AI帮助我理解最多的部分之一。AI帮助我从单条评论的情绪识别，进一步思考如何描述一个帖子下面整体评论区的情绪结构。

项目中构造的帖子层变量包括：

- 情绪效价标准差，用来表示同一帖子下评论情绪方向的分散程度。
- 主导情绪多样性，用来表示评论区是否存在多种不同的主要情绪。
- 正负情绪并存程度，用来描述评论区是否同时存在明显正向和负向表达。
- 高唤醒评论占比，用来表示评论区是否有较强烈的情绪表达。
- 负向评论占比，用来描述评论区负面情绪的集中程度。
- 情绪分布差异，用来衡量不同情绪分布之间的差异。
- 按`post_id`聚合评论层结果，使评论级情绪能够转化为帖子级极化指标。

情感极化关注同一个讨论空间内部情绪是否分化、冲突或多样化，分析单位从单条评论扩展到整个讨论串。

### 8. 传播热度指标构造

代码中的相关设计包括：

- 评论层关注评论upvotes和回复数量。
- 帖子层关注帖子upvotes、评论数、回复upvotes和综合互动。
- 对偏态严重的互动指标进行log转换，使模型更稳定。
- 构造top 20%高热度分类标签，用于机器学习二分类预测。
- 区分评论层建模数据和帖子层建模数据，避免不同分析层级混在一起。

AI 也提醒我，Reddit上的upvotes、评论数和回复数只能代表平台内部互动强度，不能直接等同于真实曝光量、社会影响力或传播范围。因此Notebook中保留了这一解释边界。

### 9. 统计建模

- 评论层模型关注负向情绪、高唤醒情绪、情绪效价、情绪强度等变量和评论热度之间的关系。
- 帖子层模型关注情绪冲突、情绪分布差异、高唤醒占比、负向占比等变量和帖子热度之间的关系。
- 模型中加入subreddit和主题控制变量，以降低社区文化和讨论议题差异带来的混淆。
- 使用连续热度指标和高热度分类指标进行不同角度的建模。
- 在解释中避免使用“导致”“影响”等强因果表述，改用“相关”“关联”“预测”等更准确的表达。

### 10. 机器学习模型

AI帮助我把传播热度问题进一步转化为预测任务，即预测评论或帖子是否进入热度前 20%。

- 比较Baseline、Logistic Regression、Random Forest和HistGradientBoosting等模型。
- 使用average precision、ROC-AUC、F1等指标评估模型表现。
- 使用统一的预处理流程处理数值特征和分类特征。
- 通过特征重要性分析理解哪些变量对预测高热度更有帮助。
- 将机器学习结果定位为“预测模式解释”，不用于因果解释。

### 11. 稳健性检验

AI帮助我补充了稳健性检验的思路。

稳健性检验包括：

- 替换高热度阈值，检查结果是否依赖top 20%的设定。
- 移除subreddit特征，检查模型是否过度依赖社区差异。
- 替换热度指标，检查不同传播热度定义下结果是否相近。
- 更换随机种子，检查机器学习结果是否对随机划分过于敏感。
- 将稳健性结果保存为表格，方便后续报告引用和复核。

AI在这里帮助我理解，好的数据分析项目不应该只展示一个最理想的模型结果，还需要检查结论是否依赖某些特定设定。

### 12. 结果可视化

- 描述统计图表来自描述统计表和建模数据。
- 系数图来自统计模型结果。
- 特征重要性图来自机器学习模型结果。
- 稳健性图来自稳健性检验结果。
- 所有正式图表统一保存到`outputs/figures`目录。
- 生成`figure_index.csv`记录图表信息，方便检查和引用。


总体来说，AI 在本项目中的作用主要是学习辅导、代码结构优化、变量构造启发、注释完善、结果表达润色。AI 帮助我把项目做得更完整、更清楚、更可复现，但研究主题、整体架构设计、数据理解、结果检查和最终解释均由我本人完成和确认。